In [1]:
from pathlib import Path
import ast
import json
import sys
from importlib.metadata import version, PackageNotFoundError

import pandas as pd

ROOT = Path("/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization")
DATA = ROOT / "data/processed/caltech_12weeks_20261111T143402Z_9b087d"

assert ROOT.is_dir(), f"项目目录不存在：{ROOT}"
assert DATA.is_dir(), f"处理数据目录不存在：{DATA}"

print("=== 1. 环境 ===")
print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
for package in ["pandas", "numpy", "pyomo", "highspy", "acnportal"]:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: 未安装")

print("\n=== 2. 已有源文件 ===")
for folder in ["src", "experiments", "tests"]:
    directory = ROOT / folder
    print(folder, sorted(p.name for p in directory.glob("*.py")))

print("\n=== 3. 数据字段与样例 ===")
for name in [
    "sessions_with_split.csv",
    "user_updates.csv",
    "environment_events.csv",
    "input_release_events.csv",
    "environment_truth.csv",
]:
    path = DATA / name
    print(f"\n--- {name} ---")
    if not path.is_file():
        print("缺失")
        continue
    frame = pd.read_csv(path)
    print("行数:", len(frame))
    print("字段:", frame.columns.tolist())
    print(frame.head(2).to_json(
        orient="records", force_ascii=False, indent=2
    ))

print("\n=== 4. 信息策略与配置 ===")
paths = [
    DATA / "information_policy.json",
    ROOT / "configs/station.yaml",
    ROOT / "configs/tariff.yaml",
    ROOT / "configs/splits.yaml",
]
for path in paths:
    print(f"\n--- {path.relative_to(ROOT)} ---")
    if path.is_file():
        print(path.read_text(encoding="utf-8"))
    else:
        print("缺失")

print("\n=== 5. 静态模型接口（解析源码，不执行模块） ===")
path = ROOT / "src/optimization_model.py"
if not path.is_file():
    print("缺失:", path)
else:
    source = path.read_text(encoding="utf-8")
    tree = ast.parse(source)
    for node in tree.body:
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            print(ast.unparse(node))
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            print(f"\n函数 {node.name}({ast.unparse(node.args)})")
            if node.returns:
                print("返回类型:", ast.unparse(node.returns))
            print("说明:", ast.get_docstring(node) or "无")
        elif isinstance(node, ast.ClassDef):
            print(f"\n类 {node.name}")
            for item in node.body:
                if isinstance(item, ast.AnnAssign):
                    print(" ", ast.unparse(item))
                elif isinstance(item, ast.FunctionDef):
                    print(f"  方法 {item.name}({ast.unparse(item.args)})")

print("\n=== 3.1 只读接口检查结束 ===")

=== 1. 环境 ===
Python: 3.11.18
解释器: /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/.envs/ev-charging/bin/python
pandas: 2.2.3
numpy: 1.26.4
pyomo: 6.10.1
highspy: 1.15.1
acnportal: 0.3.3

=== 2. 已有源文件 ===
src ['data_preprocessing.py', 'optimization_model.py', 'tariff.py']
experiments ['run_static.py']
tests ['test_static_lp.py']

=== 3. 数据字段与样例 ===

--- sessions_with_split.csv ---
行数: 2075
字段: ['sessionID', 'stationID', 'siteID', 'clusterID', 'connectionTime', 'cohort', 'initial_kWhRequested', 'initial_requestedDeparture', 'split']
[
  {
    "sessionID":"2_39_88_24_2019-05-06 13:38:12.798997",
    "stationID":"2-39-88-24",
    "siteID":2,
    "clusterID":39,
    "connectionTime":"2019-05-06 13:38:13+00:00",
    "cohort":"supplement_after_submission",
    "initial_kWhRequested":null,
    "initial_requestedDeparture":null,
    "split":"development"
  },
  {
    "sessionID":"2_39_79_379_2019-05-06 14:14:27.028419",
    "stationID":"2-39-79-379",
    "siteID":2,
    "clusterID":

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import sys

ROOT = Path("/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization")
assert ROOT.is_dir(), ROOT

simulator_source = r'''
"""Aggregate event replay. Controllers receive view() copies, never this environment."""
import math
import pandas as pd

EPS = 1e-9

def timestamp(value):
    value = pd.Timestamp(value)
    if pd.isna(value) or value.tzinfo is None:
        raise ValueError(f"Expected timezone-aware timestamp: {value}")
    return value.tz_convert("UTC")

def number(value):
    value = float(value)
    if not math.isfinite(value) or value < 0:
        raise ValueError(f"Expected finite nonnegative number: {value}")
    return value

class AggregateEnvironment:
    def __init__(self, sessions, events, updates, start,
                 capacity_kw=50.0, pmax_kw=6.656):
        self.now = timestamp(start)
        if self.now != self.now.floor("15min"):
            raise ValueError("Start must be on the 15-minute UTC grid")
        self.capacity = number(capacity_kw)
        self.pmax = number(pmax_kw)
        self._sessions = {}
        self._events = []
        self._index = 0
        self._active = {}
        self.ledger = []       # Environment/evaluator output; not controller input.
        self.completed = []
        self.update_audit = []

        for row in sessions.to_dict("records"):
            sid = str(row["sessionID"])
            if sid in self._sessions:
                raise ValueError(f"Duplicate session: {sid}")
            cohort = row["cohort"]
            if cohort not in {"main_at_arrival", "supplement_after_submission"}:
                raise ValueError(f"Unknown cohort: {cohort}")
            record = {
                "id": sid, "station_id": str(row["stationID"]),
                "arrival": timestamp(row["connectionTime"]),
                "cohort": cohort, "target_kwh": None,
                "expected_departure": None,
            }
            if cohort == "main_at_arrival":
                record["target_kwh"] = number(row["initial_kWhRequested"])
                record["expected_departure"] = timestamp(
                    row["initial_requestedDeparture"])
            if record["arrival"] < self.now:
                raise ValueError("Warm-start sessions require an explicit policy")
            self._sessions[sid] = record

        # Environment events are filtered by the selected official sessions.
        boundaries = {sid: {} for sid in self._sessions}
        for row in events.to_dict("records"):
            sid = str(row["sessionID"])
            if sid not in self._sessions:
                continue
            kind = row["event_type"]
            if kind not in {"arrival", "departure"}:
                raise ValueError(f"Unknown event type: {kind}")
            if kind in boundaries[sid]:
                raise ValueError(f"Duplicate {kind}: {sid}")
            when = timestamp(row["event_time"])
            boundaries[sid][kind] = when
            priority = 0 if kind == "departure" else 1
            self._events.append((when, priority, 0, sid, kind, None))

        for sid, bounds in boundaries.items():
            if set(bounds) != {"arrival", "departure"}:
                raise ValueError(f"Missing boundary event: {sid}")
            if bounds["arrival"] != self._sessions[sid]["arrival"]:
                raise ValueError(f"Arrival mismatch: {sid}")
            if bounds["departure"] <= bounds["arrival"]:
                raise ValueError(f"Nonpositive connection duration: {sid}")

        seen_updates = set()
        for row in updates.to_dict("records"):
            sid = str(row["sessionID"])
            if sid not in self._sessions:
                continue
            if self._sessions[sid]["cohort"] != "supplement_after_submission":
                continue                    # Main-cohort requests stay frozen.
            if str(row["valid"]).strip().lower() not in {"true", "1", "1.0"}:
                continue
            when = timestamp(row["modifiedAt"])
            order = int(row["input_order"])
            key = (sid, when, order)
            if key in seen_updates:
                raise ValueError(f"Duplicate update key: {key}")
            seen_updates.add(key)
            if when < boundaries[sid]["arrival"]:
                raise ValueError(f"Input release before arrival: {sid}")
            # Whitelist: disconnectTime and other future fields are discarded.
            payload = {
                "target_kwh": number(row["kWhRequested"]),
                "expected_departure": timestamp(row["requestedDeparture"]),
            }
            self._events.append((when, 2, order, sid, "input_update", payload))

        self._events.sort(key=lambda event: event[:4])
        self._release()

    def _release(self):
        while self._index < len(self._events):
            event = self._events[self._index]
            when, _, _, sid, kind, payload = event
            if when > self.now:
                break
            self._index += 1
            if kind == "departure":
                state = self._active.pop(sid, None)
                if state is not None:
                    self.completed.append(dict(state, actual_departure=when))
            elif kind == "arrival":
                record = self._sessions[sid]
                if any(s["station_id"] == record["station_id"]
                       for s in self._active.values()):
                    raise ValueError(f"Overlapping EVSE occupancy: {sid}")
                self._active[sid] = dict(record, delivered_kwh=0.0)
            elif sid in self._active:
                self._active[sid].update(payload)
                self.update_audit.append({
                    "time": when, "id": sid, **payload,
                    "delivered_kwh": self._active[sid]["delivered_kwh"],
                })
            # An input at/after actual departure cannot reactivate the session.

    def view(self):
        """Only already-arrived, still-connected sessions and released inputs."""
        cars = []
        for sid in sorted(self._active):
            state = self._active[sid]
            target = state["target_kwh"]
            delivered = state["delivered_kwh"]
            cars.append({
                **state,
                "pmax_kw": self.pmax,
                "remaining_kwh": (
                    None if target is None else max(target - delivered, 0.0)),
                "irreversible_excess_kwh": (
                    0.0 if target is None else max(delivered - target, 0.0)),
            })
        return {"time": self.now, "capacity_kw": self.capacity,
                "vehicles": tuple(cars)}

    def _integrate(self, boundary, command):
        while self.now < boundary:
            powers = {}
            cut = boundary
            for sid, state in self._active.items():
                target = state["target_kwh"]
                remaining = (0.0 if target is None else
                             max(target - state["delivered_kwh"], 0.0))
                power = command.get(sid, 0.0) if remaining > EPS else 0.0
                if power > 0:
                    powers[sid] = power
                    until_full = max(pd.Timedelta(hours=remaining / power),
                                     pd.Timedelta(nanoseconds=1))
                    cut = min(cut, self.now + until_full)

            hours = (cut - self.now).total_seconds() / 3600.0
            if hours <= 0:
                raise RuntimeError("Nonpositive integration interval")
            energy = {}
            actual_power = {}
            for sid, power in powers.items():
                state = self._active[sid]
                remaining = max(
                    state["target_kwh"] - state["delivered_kwh"], 0.0)
                amount = min(power * hours, remaining)
                state["delivered_kwh"] += amount
                energy[sid] = amount
                actual_power[sid] = amount / hours

            self.ledger.append({
                "start": self.now, "end": cut,
                "power_kw": actual_power,
                "energy_kwh": energy,
                "total_kw": sum(actual_power.values()),
            })
            self.now = cut

    def step(self, command):
        """Hold this first-slot command for 15 minutes; events can stop power."""
        command = {str(sid): number(power) for sid, power in command.items()}
        for sid, power in command.items():
            if sid not in self._active:
                raise ValueError(f"Command for a nonactive session: {sid}")
            if power > self.pmax + EPS:
                raise ValueError(f"Device power exceeded: {sid}")
            if power > 0 and self._active[sid]["target_kwh"] is None:
                raise ValueError(f"No released request: {sid}")
        if sum(command.values()) > self.capacity + EPS:
            raise ValueError("Station capacity exceeded")

        end = self.now + pd.Timedelta(minutes=15)
        while self.now < end:
            next_event = (
                self._events[self._index][0]
                if self._index < len(self._events) else end)
            boundary = min(end, next_event)
            if boundary > self.now:
                self._integrate(boundary, command)
            self._release()
        return self.view()
'''

test_source = r'''
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
from src.simulator import AggregateEnvironment

T = pd.Timestamp("2019-05-06T07:00:00Z")
passed = []

def check(name, condition):
    if not condition:
        raise AssertionError(name)
    passed.append(name)

def fixture(cohort="main_at_arrival", target=10.0, departure=60,
            updates=None, arrival=0):
    sessions = pd.DataFrame([{
        "sessionID": "a", "stationID": "evse-a",
        "connectionTime": T + pd.Timedelta(minutes=arrival),
        "cohort": cohort, "initial_kWhRequested": target,
        "initial_requestedDeparture": T + pd.Timedelta(hours=1),
    }])
    events = pd.DataFrame([
        {"sessionID": "a", "event_type": "arrival",
         "event_time": T + pd.Timedelta(minutes=arrival)},
        {"sessionID": "a", "event_type": "departure",
         "event_time": T + pd.Timedelta(minutes=departure)},
    ])
    records = []
    for minute, demand, order in (updates or []):
        records.append({
            "sessionID": "a", "modifiedAt": T + pd.Timedelta(minutes=minute),
            "input_order": order, "valid": True, "kWhRequested": demand,
            "requestedDeparture": T + pd.Timedelta(hours=1),
            "disconnectTime": T + pd.Timedelta(minutes=departure),
        })
    return sessions, events, pd.DataFrame(records)

def env(**kwargs):
    return AggregateEnvironment(*fixture(**kwargs), start=T)

e = env()
e.step({"a": 6.0})
check("15-minute energy integral",
      abs(e.view()["vehicles"][0]["delivered_kwh"] - 1.5) < 1e-8)

e = env(departure=7.5)
e.step({"a": 6.0})
check("mid-slot departure stops power",
      not e.view()["vehicles"] and
      abs(e.completed[0]["delivered_kwh"] - 0.75) < 1e-8)

e = env(target=1.0)
e.step({"a": 6.0})
check("stop at target without overcharging",
      abs(e.view()["vehicles"][0]["delivered_kwh"] - 1.0) < 1e-8)

e = env(cohort="supplement_after_submission", updates=[(5, 10, 0)])
check("late input initially hidden",
      e.view()["vehicles"][0]["target_kwh"] is None)
try:
    e.step({"a": 6.0})
except ValueError:
    check("no charging before released request", e.now == T)
else:
    raise AssertionError("Unreleased request accepted")
e.step({})
check("late input released without retroactive energy",
      e.view()["vehicles"][0]["target_kwh"] == 10 and
      e.view()["vehicles"][0]["delivered_kwh"] == 0)

e = env(cohort="supplement_after_submission",
        updates=[(0, 10, 0), (7.5, 0.5, 1)])
e.step({"a": 6.0})
car = e.view()["vehicles"][0]
check("downward update stops power and preserves irreversible excess",
      abs(car["delivered_kwh"] - 0.75) < 1e-8 and
      abs(car["irreversible_excess_kwh"] - 0.25) < 1e-8)

e = env(cohort="supplement_after_submission", departure=15,
        updates=[(0, 10, 0), (15, 20, 1)])
e.step({"a": 6.0})
check("departure precedes simultaneous update",
      not e.view()["vehicles"] and e.completed[0]["target_kwh"] == 10)

e = env(cohort="supplement_after_submission",
        updates=[(0, 10, 0), (0, 12, 1)])
check("same-time last input_order wins",
      e.view()["vehicles"][0]["target_kwh"] == 12)

e = env(updates=[(0, 99, 0)])
check("main cohort ignores subsequent changes",
      e.view()["vehicles"][0]["target_kwh"] == 10)

a, b = env(departure=30), env(departure=60)
check("future departure hidden from current view", a.view() == b.view())
v = a.view()
v["vehicles"][0]["target_kwh"] = 999
check("view mutation cannot change environment",
      a.view()["vehicles"][0]["target_kwh"] == 10)

e = env(arrival=5)
check("future arrival hidden", not e.view()["vehicles"])
e.step({})
check("mid-slot arrival waits until next decision",
      e.view()["vehicles"][0]["delivered_kwh"] == 0)

e = env()
try:
    e.step({"a": 7.0})
except ValueError:
    check("invalid device command rejected before advancing", e.now == T)
else:
    raise AssertionError("Device violation accepted")

e = AggregateEnvironment(*fixture(), start=T, capacity_kw=5)
try:
    e.step({"a": 6.0})
except ValueError:
    check("invalid station command rejected before advancing", e.now == T)
else:
    raise AssertionError("Station violation accepted")

# Cross-local-midnight execution remains continuous.
s, events, updates = fixture(target=10, departure=60)
offset = pd.Timedelta(hours=23, minutes=45)
s["connectionTime"] = s["connectionTime"] + offset
s["initial_requestedDeparture"] = s["initial_requestedDeparture"] + offset
events["event_time"] = events["event_time"] + offset
e = AggregateEnvironment(s, events, updates, start=T + offset)
e.step({"a": 6})
e.step({"a": 6})
check("local midnight preserves delivered energy",
      abs(e.view()["vehicles"][0]["delivered_kwh"] - 3) < 1e-8)

integrated = sum(
    row["total_kw"] *
    (row["end"] - row["start"]).total_seconds() / 3600
    for row in e.ledger)
check("ledger independently conserves energy", abs(integrated - 3) < 1e-8)

# Actual-data zero-command replay: development only, cohorts separately.
data = ROOT / "data/processed/caltech_12weeks_20261111T143402Z_9b087d"
sessions = pd.read_csv(data / "sessions_with_split.csv")
events = pd.read_csv(data / "environment_events.csv")
updates = pd.read_csv(data / "input_release_events.csv")
counts = {}
for cohort, expected_count in [
    ("main_at_arrival", 64),
    ("supplement_after_submission", 1227),
]:
    selected = sessions[
        (sessions["split"] == "development") &
        (sessions["cohort"] == cohort)
    ]
    check(f"{cohort}: selected count", len(selected) == expected_count)
    e = AggregateEnvironment(selected, events, updates, start=T)
    end = pd.Timestamp("2019-06-24T07:00:00Z")
    while e.now < end:
        e.step({})
    check(f"{cohort}: all sessions depart with zero delivered",
          len(e.completed) == expected_count and
          not e.view()["vehicles"] and
          all(s["delivered_kwh"] == 0 for s in e.completed))
    counts[cohort] = len(e.completed)

result = {
    "status": "passed", "checks": passed, "check_count": len(passed),
    "development_zero_replay_counts": counts,
    "scope": "aggregate environment only; ACN-Sim adapter not yet validated",
}
output = Path(sys.argv[1])
output.write_text(json.dumps(result, ensure_ascii=False, indent=2),
                  encoding="utf-8")
print(json.dumps(result, ensure_ascii=False, indent=2))
'''

files = {
    ROOT / "src/simulator.py": simulator_source.lstrip(),
    ROOT / "tests/test_event_environment.py": test_source.lstrip(),
}

# Preflight all targets before writing anything.
for path, text in files.items():
    compile(text, str(path), "exec")
    if path.exists() and path.read_text(encoding="utf-8") != text:
        raise FileExistsError(f"保留已有文件，未覆盖：{path}")

for path, text in files.items():
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
log_dir = ROOT / "results/logs" / f"stage3_environment_{stamp}"
log_dir.mkdir(parents=True, exist_ok=False)

command = [
    sys.executable,
    str(ROOT / "tests/test_event_environment.py"),
    str(log_dir / "verification.json"),
]
with (log_dir / "stdout_stderr.txt").open("w", encoding="utf-8") as stream:
    try:
        result = subprocess.run(
            command, cwd=ROOT, stdout=stream,
            stderr=subprocess.STDOUT, text=True, timeout=180,
        )
        exit_code = result.returncode
    except subprocess.TimeoutExpired:
        exit_code = "TIMEOUT"

print("解释器：", sys.executable)
print("退出码：", exit_code)
print("证据目录：", log_dir)
print((log_dir / "stdout_stderr.txt").read_text(encoding="utf-8"))

解释器： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/.envs/ev-charging/bin/python
退出码： 0
证据目录： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/results/logs/stage3_environment_20261112T073225916210Z
{
  "status": "passed",
  "checks": [
    "15-minute energy integral",
    "mid-slot departure stops power",
    "stop at target without overcharging",
    "late input initially hidden",
    "no charging before released request",
    "late input released without retroactive energy",
    "downward update stops power and preserves irreversible excess",
    "departure precedes simultaneous update",
    "same-time last input_order wins",
    "main cohort ignores subsequent changes",
    "future departure hidden from current view",
    "view mutation cannot change environment",
    "future arrival hidden",
    "mid-slot arrival waits until next decision",
    "invalid device command rejected before advancing",
    "invalid station command rejected before advancing",
    "loc

In [3]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import sys
import shutil

ROOT = Path("/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
LOG = ROOT / "results/logs" / f"stage3_execution_checks_{stamp}"
LOG.mkdir(parents=True, exist_ok=False)

# 精确替换已知语句；保留修正前源码。
path = ROOT / "src/simulator.py"
source = path.read_text(encoding="utf-8")
old = "hours = (cut - self.now).total_seconds() / 3600.0"
new = "hours = (cut - self.now).value / 3_600_000_000_000"

if old in source:
    assert source.count(old) == 1, "匹配不唯一，停止修改"
    shutil.copy2(path, LOG / "simulator_before.py")
    revised = source.replace(old, new)
    compile(revised, str(path), "exec")
    path.write_text(revised, encoding="utf-8")
elif new not in source:
    raise RuntimeError("源码与预期不同，未修改")

test_source = r'''
from pathlib import Path
import inspect
import json
import sys
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
from src.simulator import AggregateEnvironment
from acnportal.acnsim.models import Battery, EV, EVSE
from acnportal.acnsim.network import ChargingNetwork

T = pd.Timestamp("2019-05-06T07:00:00Z")
passed = []

def check(name, condition):
    if not condition:
        raise AssertionError(name)
    passed.append(name)

def make(rows, capacity=10):
    # rows: (id, station, arrival_minutes, departure_minutes, target)
    sessions, events = [], []
    for sid, station, arrival, departure, target in rows:
        sessions.append({
            "sessionID": sid, "stationID": station,
            "connectionTime": T + pd.Timedelta(minutes=arrival),
            "cohort": "main_at_arrival",
            "initial_kWhRequested": target,
            "initial_requestedDeparture": T + pd.Timedelta(hours=2),
        })
        for kind, minute in [("arrival", arrival), ("departure", departure)]:
            events.append({
                "sessionID": sid, "event_type": kind,
                "event_time": T + pd.Timedelta(minutes=minute),
            })
    return AggregateEnvironment(
        pd.DataFrame(sessions), pd.DataFrame(events),
        pd.DataFrame(), start=T, capacity_kw=capacity)

def delivered(environment, sid):
    for state in environment.view()["vehicles"]:
        if state["id"] == sid:
            return state["delivered_kwh"]
    return next(s["delivered_kwh"] for s in environment.completed
                if s["id"] == sid)

e = make([("a", "A", 0, 60, 10), ("b", "B", 0, 60, 10)])
try:
    e.step({"a": 6, "b": 6})
except ValueError:
    check("shared-capacity excess rejected without state change",
          e.now == T and delivered(e, "a") == 0 and delivered(e, "b") == 0)
else:
    raise AssertionError("Shared-capacity violation accepted")

e.step({"a": 5, "b": 5})
check("two vehicles execute within shared capacity",
      abs(delivered(e, "a") - 1.25) < 1e-8 and
      abs(delivered(e, "b") - 1.25) < 1e-8)
check("every executed segment respects station capacity",
      all(row["total_kw"] <= 10 + 1e-8 for row in e.ledger))

e = make([("a", "A", 0, 60, 0.5), ("b", "B", 0, 60, 10)])
e.step({"a": 6, "b": 4})
check("one vehicle completing does not stop another",
      abs(delivered(e, "a") - 0.5) < 1e-8 and
      abs(delivered(e, "b") - 1.0) < 1e-8)
check("target completion preserves realized peak",
      abs(max(row["total_kw"] for row in e.ledger) - 10) < 1e-8)

integral = sum(
    row["total_kw"] * (row["end"] - row["start"]).value
    / 3_600_000_000_000
    for row in e.ledger)
check("multi-vehicle ledger independently conserves energy",
      abs(integral - 1.5) < 1e-8)

e = make([("a", "A", 0, 7.5, 10), ("b", "A", 7.5, 60, 10)])
e.step({"a": 6})
check("same-EVSE handoff does not inherit prior vehicle command",
      abs(delivered(e, "a") - 0.75) < 1e-8 and
      delivered(e, "b") == 0)
e.step({"b": 6})
check("new session charges at next control boundary",
      abs(delivered(e, "b") - 1.5) < 1e-8)

e = make([("a", "A", 0, 60, 1.5e-9)])
e.step({"a": 6})
check("sub-microsecond completion integrates without zero-time loop",
      abs(delivered(e, "a") - 1.5e-9) < 1e-12)

e = make([("a", "A", 0, 60, 0.123456789)])
e.step({"a": 6.656})
check("fractional target completion remains accurate",
      abs(delivered(e, "a") - 0.123456789) < 1e-8)

# Synthetic bookkeeping batteries only: these are not inferred real SOC/capacity.
def device(max_power):
    battery = Battery(capacity=100.0, init_charge=0.0,
                      max_power=max_power)
    ev = EV(arrival=0, departure=100, requested_energy=10.0,
            station_id="test", session_id="synthetic",
            battery=battery)
    evse = EVSE("test", max_rate=32.0, min_rate=0.0)
    evse.plugin(ev)
    return ev, evse

ev, evse = device(6.656)
evse.set_pilot(32.0, 208.0, 15.0)
check("ACN-Sim ideal response: A/V/min to kWh",
      abs(ev.current_charging_rate - 32) < 1e-8 and
      abs(ev.energy_delivered - 1.664) < 1e-8)

ev, evse = device(3.0)
evse.set_pilot(32.0, 208.0, 15.0)
check("ACN-Sim actual delivery reflects device power limit",
      abs(ev.energy_delivered - 0.75) < 1e-8 and
      ev.current_charging_rate < 32)

before = ev.energy_delivered
evse.set_pilot(0.0, 208.0, 15.0)
check("ACN-Sim zero pilot adds zero energy",
      abs(ev.energy_delivered - before) < 1e-10)

# Capture installed-version interfaces needed for the network adapter.
interfaces = {}
for obj in [Battery, EV, EVSE, ChargingNetwork,
            ChargingNetwork.register_evse,
            ChargingNetwork.plugin, ChargingNetwork.unplug,
            ChargingNetwork.update_pilots,
            ChargingNetwork.is_feasible]:
    interfaces[obj.__qualname__] = str(inspect.signature(obj))

network_source = inspect.getsource(ChargingNetwork.update_pilots)
ev_charge_source = inspect.getsource(EV.charge)

output = Path(sys.argv[1])
result = {
    "status": "passed",
    "check_count": len(passed),
    "checks": passed,
    "installed_interfaces": interfaces,
    "scope": "aggregate execution and isolated ACN-Sim device checks; "
             "full network adapter not yet validated",
}
output.write_text(json.dumps(result, ensure_ascii=False, indent=2),
                  encoding="utf-8")
print(json.dumps(result, ensure_ascii=False, indent=2))
print("\n=== Installed ChargingNetwork.update_pilots ===")
print(network_source)
print("\n=== Installed EV.charge ===")
print(ev_charge_source)
'''

test_path = ROOT / "tests/test_execution_and_acnsim.py"
text = test_source.lstrip()
compile(text, str(test_path), "exec")
if test_path.exists() and test_path.read_text(encoding="utf-8") != text:
    raise FileExistsError(f"保留已有文件，未覆盖：{test_path}")
test_path.write_text(text, encoding="utf-8")

print("解释器：", sys.executable)
print("证据目录：", LOG)

for name, script in [
    ("regression", ROOT / "tests/test_event_environment.py"),
    ("execution_acnsim", test_path),
]:
    log_path = LOG / f"{name}_stdout_stderr.txt"
    with log_path.open("w", encoding="utf-8") as stream:
        try:
            result = subprocess.run(
                [sys.executable, str(script),
                 str(LOG / f"{name}_verification.json")],
                cwd=ROOT, stdout=stream, stderr=subprocess.STDOUT,
                text=True, timeout=180,
            )
            code = result.returncode
        except subprocess.TimeoutExpired:
            code = "TIMEOUT"
    print(f"\n=== {name}：退出码 {code} ===")
    print(log_path.read_text(encoding="utf-8"))
    if code != 0:
        print("本轮停止，保留日志供核对。")
        break

解释器： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/.envs/ev-charging/bin/python
证据目录： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/results/logs/stage3_execution_checks_20261112T073549304156Z

=== regression：退出码 0 ===
{
  "status": "passed",
  "checks": [
    "15-minute energy integral",
    "mid-slot departure stops power",
    "stop at target without overcharging",
    "late input initially hidden",
    "no charging before released request",
    "late input released without retroactive energy",
    "downward update stops power and preserves irreversible excess",
    "departure precedes simultaneous update",
    "same-time last input_order wins",
    "main cohort ignores subsequent changes",
    "future departure hidden from current view",
    "view mutation cannot change environment",
    "future arrival hidden",
    "mid-slot arrival waits until next decision",
    "invalid device command rejected before advancing",
    "invalid station command rejected bef

In [4]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import sys
import shutil

ROOT = Path("/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
LOG = ROOT / "results/logs" / f"stage3_adapter_{stamp}"
LOG.mkdir(parents=True, exist_ok=False)

adapter_source = r'''
"""ACN-Sim component backend for the aggregate continuous-power scenario.

Event timing remains in AggregateEnvironment.
Synthetic Battery capacity is bookkeeping, not inferred physical capacity/SOC.
All EVSEs use an artificial common phase to express aggregate kW capacity.
Original Caltech branch/phase constraints are not represented here.
"""
import math
import numpy as np
from acnportal.acnsim.models import Battery, EV, EVSE
from acnportal.acnsim.network import ChargingNetwork, Current

class ACNSimExecutor:
    def __init__(self, station_ids, capacity_kw=50.0, voltage=208.0,
                 max_current=32.0, response_max_kw=None):
        self.ids = sorted(set(str(s) for s in station_ids))
        self.capacity_kw = float(capacity_kw)
        self.voltage = float(voltage)
        self.max_current = float(max_current)
        for value in [self.capacity_kw, self.voltage, self.max_current]:
            if not math.isfinite(value) or value <= 0:
                raise ValueError("Capacity, voltage and current must be positive")
        if not self.ids:
            raise ValueError("At least one EVSE is required")
        nominal = self.voltage * self.max_current / 1000
        self.response_max_kw = (
            nominal if response_max_kw is None else float(response_max_kw))
        if (not math.isfinite(self.response_max_kw) or
                not 0 < self.response_max_kw <= nominal):
            raise ValueError("Invalid synthetic response limit")

        self.network = ChargingNetwork()
        for station in self.ids:
            self.network.register_evse(
                EVSE(station, max_rate=self.max_current, min_rate=0.0),
                voltage=self.voltage, phase_angle=0.0)
        self.network.add_constraint(
            Current({station: 1.0 for station in self.ids}),
            self.capacity_kw * 1000 / self.voltage,
            name="aggregate_experimental_capacity")

        self._connected = {}
        self.evs = {}       # Environment/evaluator access only.
        self.records = []

    def sync(self, view):
        vehicles = view["vehicles"]
        desired = {v["station_id"]: v["id"] for v in vehicles}
        if len(desired) != len(vehicles):
            raise ValueError("Multiple sessions occupy the same EVSE")
        if set(desired) - set(self.ids):
            raise ValueError("Unregistered EVSE")

        for station, sid in list(self._connected.items()):
            if desired.get(station) != sid:
                self.network.unplug(station, session_id=sid)
                del self._connected[station]

        for station, sid in desired.items():
            if station in self._connected:
                continue
            if sid in self.evs:
                raise ValueError("Unexpected repeated session connection")
            # Bookkeeping reservoir; demand limits are enforced by the environment.
            reservoir = 1_000_000.0
            battery = Battery(
                capacity=reservoir, init_charge=0.0,
                max_power=self.response_max_kw)
            ev = EV(
                arrival=0, departure=1_000_000_000,
                requested_energy=reservoir,
                station_id=station, session_id=sid, battery=battery)
            self.network.plugin(ev)
            self.evs[sid] = ev
            self._connected[station] = sid

    def execute(self, view, powers_kw, hours):
        self.sync(view)
        if not math.isfinite(hours) or hours <= 0:
            raise ValueError("Invalid execution duration")
        station_for = {v["id"]: v["station_id"] for v in view["vehicles"]}
        pilots = np.zeros((len(self.ids), 1))
        index = {station: i for i, station in enumerate(self.ids)}

        for sid, value in powers_kw.items():
            power = float(value)
            if sid not in station_for:
                raise ValueError("Command for a nonactive session")
            if not math.isfinite(power) or power < 0:
                raise ValueError("Invalid power")
            current = power * 1000 / self.voltage
            if current > self.max_current + 1e-8:
                raise ValueError("EVSE current exceeded")
            pilots[index[station_for[sid]], 0] = current

        # update_pilots itself does not enforce network constraints.
        if not self.network.is_feasible(pilots):
            raise ValueError("ACN-Sim network rejected the requested pilots")

        self.network.update_pilots(pilots, i=0, period=hours * 60)
        actual = {
            sid: float(self.evs[sid].current_charging_rate)
                 * self.voltage / 1000
            for sid in powers_kw
        }
        self.records.append({
            "time": view["time"], "hours": hours,
            "requested_kw": dict(powers_kw), "actual_kw": dict(actual),
            "network_feasible": True,
        })
        return actual
'''

# Add an optional executor without changing the default ideal backend.
sim_path = ROOT / "src/simulator.py"
original = sim_path.read_text(encoding="utf-8")
revised = original

if "self._executor = executor" not in original:
    replacements = [
        (
            "capacity_kw=50.0, pmax_kw=6.656):",
            "capacity_kw=50.0, pmax_kw=6.656, executor=None):"
        ),
        (
            "        self.now = timestamp(start)",
            "        self._executor = executor\n"
            "        self.now = timestamp(start)"
        ),
        (
            "    def view(self):",
            "        if self._executor is not None:\n"
            "            self._executor.sync(self.view())\n\n"
            "    def view(self):"
        ),
        (
            "            energy = {}\n"
            "            actual_power = {}\n"
            "            for sid, power in powers.items():",
            "            executed = (powers if self._executor is None else\n"
            "                        self._executor.execute(self.view(), powers, hours))\n"
            "            if set(executed) - set(powers):\n"
            "                raise ValueError('Unexpected executor session')\n"
            "            executed = {sid: number(p) for sid, p in executed.items()}\n"
            "            if any(p > powers[sid] + 1e-8\n"
            "                   for sid, p in executed.items()):\n"
            "                raise ValueError('Actual power exceeds requested power')\n"
            "            energy = {}\n"
            "            actual_power = {}\n"
            "            for sid, power in executed.items():"
        ),
    ]
    for before, after in replacements:
        if revised.count(before) != 1:
            raise RuntimeError(f"源码匹配不唯一，未写入：{before[:100]}")
        revised = revised.replace(before, after)

assert "hours = (cut - self.now).value / 3_600_000_000_000" in revised

test_source = r'''
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
from src.simulator import AggregateEnvironment
from src.acnsim_adapter import ACNSimExecutor

T = pd.Timestamp("2019-05-06T07:00:00Z")
passed = []

def check(name, condition):
    if not condition:
        raise AssertionError(name)
    passed.append(name)

def build(rows, updates=(), backend=False, response=None):
    # rows: id, station, arrival_minute, departure_minute, demand, cohort
    sessions, events, releases = [], [], []
    for sid, station, arrival, departure, demand, cohort in rows:
        sessions.append({
            "sessionID": sid, "stationID": station,
            "connectionTime": T + pd.Timedelta(minutes=arrival),
            "cohort": cohort, "initial_kWhRequested": demand,
            "initial_requestedDeparture": T + pd.Timedelta(hours=2),
        })
        for kind, minute in [("arrival", arrival), ("departure", departure)]:
            events.append({
                "sessionID": sid, "event_type": kind,
                "event_time": T + pd.Timedelta(minutes=minute),
            })
    for sid, minute, demand, order in updates:
        releases.append({
            "sessionID": sid, "modifiedAt": T + pd.Timedelta(minutes=minute),
            "input_order": order, "valid": True, "kWhRequested": demand,
            "requestedDeparture": T + pd.Timedelta(hours=2),
        })
    executor = (
        ACNSimExecutor([row[1] for row in rows], capacity_kw=10,
                       response_max_kw=response) if backend else None)
    environment = AggregateEnvironment(
        pd.DataFrame(sessions), pd.DataFrame(events), pd.DataFrame(releases),
        start=T, capacity_kw=10, executor=executor)
    return environment, executor

def command(view):
    # Deterministic diagnostic control only; not a formal baseline.
    remaining_capacity = view["capacity_kw"]
    result = {}
    for v in view["vehicles"]:
        if v["remaining_kwh"] is not None and v["remaining_kwh"] > 1e-9:
            power = min(6.0, remaining_capacity)
            if power > 0:
                result[v["id"]] = power
                remaining_capacity -= power
    return result

def totals(e):
    return {s["id"]: s["delivered_kwh"] for s in e.completed}

main = "main_at_arrival"
supplement = "supplement_after_submission"
scenarios = [
    ("two vehicles",
     [("a", "A", 0, 60, 10, main), ("b", "B", 0, 60, 10, main)], []),
    ("mid-slot departure",
     [("a", "A", 0, 7.5, 10, main)], []),
    ("late input",
     [("a", "A", 0, 60, None, supplement)], [("a", 5, 10, 0)]),
    ("downward update",
     [("a", "A", 0, 60, None, supplement)],
     [("a", 0, 10, 0), ("a", 7.5, 0.5, 1)]),
    ("same-EVSE handoff",
     [("a", "A", 0, 7.5, 10, main), ("b", "A", 7.5, 60, 10, main)], []),
    ("fractional completion",
     [("a", "A", 0, 60, 0.123456789, main)], []),
]

for name, rows, updates in scenarios:
    ideal, _ = build(rows, updates)
    adapted, executor = build(rows, updates, backend=True)
    for _ in range(4):
        ideal.step(command(ideal.view()))
        adapted.step(command(adapted.view()))
    a, b = totals(ideal), totals(adapted)
    check(f"{name}: ideal and ACN-Sim delivery agree",
          a.keys() == b.keys() and
          all(abs(a[sid] - b[sid]) < 1e-7 for sid in a))
    check(f"{name}: environment agrees with ACN-Sim cumulative feedback",
          all(abs(b[sid] - executor.evs[sid].energy_delivered) < 1e-7
              for sid in b))
    check(f"{name}: no final connection or capacity violation",
          not executor._connected and
          all(row["total_kw"] <= 10 + 1e-8 for row in adapted.ledger))

e, executor = build(
    [("a", "A", 0, 60, 10, main)], backend=True, response=3.0)
e.step({"a": 6.0})
actual = e.view()["vehicles"][0]["delivered_kwh"]
check("reduced response updates environment with actual energy",
      abs(actual - 0.75) < 1e-8 and
      abs(executor.evs["a"].energy_delivered - actual) < 1e-8)

# Test the adapter's own network gate independently of the environment gate.
e, executor = build(
    [("a", "A", 0, 60, 10, main), ("b", "B", 0, 60, 10, main)],
    backend=True)
try:
    executor.execute(e.view(), {"a": 6, "b": 6}, 0.25)
except ValueError:
    check("network rejects excessive pilots before energy execution",
          all(ev.energy_delivered == 0 for ev in executor.evs.values()))
else:
    raise AssertionError("Network accepted excessive aggregate current")

result = {
    "status": "passed", "check_count": len(passed), "checks": passed,
    "backend": "ACN-Sim network/EVSE/EV/Battery components",
    "event_engine": "project AggregateEnvironment",
    "network_scope": "aggregate experimental capacity; common artificial phase",
    "physical_battery_parameters_inferred": False,
    "original_caltech_branch_constraints_validated": False,
}
Path(sys.argv[1]).write_text(
    json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(result, ensure_ascii=False, indent=2))
'''

files = {
    ROOT / "src/acnsim_adapter.py": adapter_source.lstrip(),
    ROOT / "tests/test_acnsim_adapter.py": test_source.lstrip(),
}

# Validate all proposed sources before mutation.
compile(revised, str(sim_path), "exec")
for path, text in files.items():
    compile(text, str(path), "exec")
    if path.exists() and path.read_text(encoding="utf-8") != text:
        raise FileExistsError(f"保留已有不同文件，未覆盖：{path}")

shutil.copy2(sim_path, LOG / "simulator_before.py")
sim_path.write_text(revised, encoding="utf-8")
for path, text in files.items():
    path.write_text(text, encoding="utf-8")

print("解释器：", sys.executable)
print("证据目录：", LOG)

for name, filename in [
    ("event_regression", "test_event_environment.py"),
    ("execution_regression", "test_execution_and_acnsim.py"),
    ("adapter_integration", "test_acnsim_adapter.py"),
]:
    output = LOG / f"{name}_stdout_stderr.txt"
    with output.open("w", encoding="utf-8") as stream:
        try:
            result = subprocess.run(
                [sys.executable, str(ROOT / "tests" / filename),
                 str(LOG / f"{name}_verification.json")],
                cwd=ROOT, stdout=stream, stderr=subprocess.STDOUT,
                text=True, timeout=180,
            )
            code = result.returncode
        except subprocess.TimeoutExpired:
            code = "TIMEOUT"
    print(f"\n=== {name}：退出码 {code} ===")
    print(output.read_text(encoding="utf-8"))
    if code != 0:
        print("本轮停止，保留源码备份及日志。")
        break

解释器： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/.envs/ev-charging/bin/python
证据目录： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/results/logs/stage3_adapter_20261112T074129922228Z

=== event_regression：退出码 0 ===
{
  "status": "passed",
  "checks": [
    "15-minute energy integral",
    "mid-slot departure stops power",
    "stop at target without overcharging",
    "late input initially hidden",
    "no charging before released request",
    "late input released without retroactive energy",
    "downward update stops power and preserves irreversible excess",
    "departure precedes simultaneous update",
    "same-time last input_order wins",
    "main cohort ignores subsequent changes",
    "future departure hidden from current view",
    "view mutation cannot change environment",
    "future arrival hidden",
    "mid-slot arrival waits until next decision",
    "invalid device command rejected before advancing",
    "invalid station command rejected before

In [7]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import sys

ROOT = Path("/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
LOG = ROOT / "results/logs" / f"stage3_environment_acceptance_{stamp}"
LOG.mkdir(parents=True, exist_ok=False)

source = r'''
from pathlib import Path
from bisect import bisect_right
import hashlib
import json
import sys
import shutil
import pandas as pd
import yaml

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
from src.simulator import AggregateEnvironment
from src.acnsim_adapter import ACNSimExecutor

OUT = Path(sys.argv[1])
DATA = ROOT / "data/processed/caltech_12weeks_20261111T143402Z_9b087d"
TZ = "America/Los_Angeles"
DT = pd.Timedelta(minutes=15)
passed = []

def check(name, condition):
    if not condition:
        raise AssertionError(name)
    passed.append(name)

def hours(delta):
    return delta.value / 3_600_000_000_000

def valid(value):
    return str(value).strip().lower() in {"true", "1", "1.0"}

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

station = yaml.safe_load(
    (ROOT / "configs/station.yaml").read_text(encoding="utf-8"))
splits = yaml.safe_load(
    (ROOT / "configs/splits.yaml").read_text(encoding="utf-8"))
check("frozen aggregate scope",
      station["network_model"] == "aggregate_continuous" and
      not station["original_branch_constraints_enforced"] and
      station["control_interval_minutes"] == 15 and
      abs(station["charging_efficiency"] - 1) < 1e-12)

capacity = float(station["station_capacity_kW"])
pmax = float(station["evse_max_power_kW"])
voltage = float(station["evse_voltage_V"])
current = float(station["evse_max_current_A"])
check("device power and current units agree",
      abs(pmax - voltage * current / 1000) < 1e-10)

start = pd.Timestamp(splits["splits"]["development"]["start_utc_inclusive"])
dev_end = pd.Timestamp(splits["splits"]["development"]["end_utc_exclusive"])
arrival_end = start + pd.Timedelta(days=7)

sessions = pd.read_csv(DATA / "sessions_with_split.csv")
events = pd.read_csv(DATA / "environment_events.csv")
releases = pd.read_csv(DATA / "input_release_events.csv")
independent_updates = pd.read_csv(DATA / "user_updates.csv")
sessions["connectionTime"] = pd.to_datetime(
    sessions["connectionTime"], utc=True)
events["event_time"] = pd.to_datetime(events["event_time"], utc=True)

# Diagnostic control only. It reads view() and has no environment/file access.
def diagnostic_control(view):
    result = {}
    capacity_left = view["capacity_kw"]
    now = view["time"]
    vehicles = sorted(
        view["vehicles"], key=lambda v: (v["arrival"], v["id"]))
    for v in vehicles:
        remaining = v["remaining_kwh"]
        if remaining is None or remaining <= 1e-9:
            continue
        estimate = v["expected_departure"]
        deadline = (now + DT if estimate <= now else estimate.floor("15min"))
        if deadline < now + DT:
            continue
        power = min(v["pmax_kw"], capacity_left, remaining / 0.25)
        if power > 0:
            result[v["id"]] = power
            capacity_left -= power
    return result

def run(selected, selected_events, selected_releases, begin, end,
        backend, run_dir, control=diagnostic_control):
    executor = None
    if backend:
        executor = ACNSimExecutor(
            selected["stationID"].astype(str).tolist(),
            capacity_kw=capacity, voltage=voltage, max_current=current)
    e = AggregateEnvironment(
        selected, selected_events, selected_releases, start=begin,
        capacity_kw=capacity, pmax_kw=pmax, executor=executor)
    while e.now < end:
        e.step(control(e.view()))
    if run_dir is not None:
        run_dir.mkdir(parents=True, exist_ok=True)
        station_rows, energy_rows = [], []
        for row in e.ledger:
            station_rows.append({
                "start": row["start"], "end": row["end"],
                "total_kw": row["total_kw"],
                "energy_kwh": sum(row["energy_kwh"].values()),
            })
            for sid, energy in row["energy_kwh"].items():
                energy_rows.append({
                    "sessionID": sid, "start": row["start"], "end": row["end"],
                    "actual_kw": row["power_kw"][sid], "energy_kwh": energy,
                })
        pd.DataFrame(station_rows).to_csv(
            run_dir / "station_trajectory.csv", index=False)
        pd.DataFrame(energy_rows).to_csv(
            run_dir / "session_trajectory.csv", index=False)
        pd.DataFrame(e.completed).to_csv(
            run_dir / "completed_sessions.csv", index=False)
    return e, executor

def audit(e, executor, selected, selected_events, prefix):
    arrivals = dict(zip(selected["sessionID"], selected["connectionTime"]))
    departures = dict(
        selected_events.loc[
            selected_events["event_type"] == "departure",
            ["sessionID", "event_time"]].itertuples(index=False, name=None))
    check(prefix + ": complete selected-session replay",
          len(e.completed) == len(selected) and not e.view()["vehicles"])

    # Build an independent request history from user_updates.csv.
    history = {}
    for row in selected.to_dict("records"):
        sid = row["sessionID"]
        if row["cohort"] == "main_at_arrival":
            history[sid] = (
                [arrivals[sid].value], [float(row["initial_kWhRequested"])])
        else:
            u = independent_updates[
                independent_updates["sessionID"] == sid].copy()
            u = u[u["valid"].map(valid)]
            u["modifiedAt"] = pd.to_datetime(u["modifiedAt"], utc=True)
            u = u[(u["modifiedAt"] >= arrivals[sid]) &
                  (u["modifiedAt"] < departures[sid])]
            u = u.sort_values(["modifiedAt", "input_order"])
            history[sid] = (
                [t.value for t in u["modifiedAt"]],
                u["kWhRequested"].astype(float).tolist())

    cumulative = {sid: 0.0 for sid in arrivals}
    max_residual = 0.0
    capacity_residual = 0.0
    device_residual = 0.0
    for segment in e.ledger:
        h = hours(segment["end"] - segment["start"])
        assert h > 0
        capacity_residual = max(
            capacity_residual, segment["total_kw"] - capacity)
        max_residual = max(
            max_residual,
            abs(sum(segment["energy_kwh"].values()) - segment["total_kw"] * h))
        for sid, energy in segment["energy_kwh"].items():
            power = segment["power_kw"][sid]
            device_residual = max(device_residual, power - pmax, -power)
            max_residual = max(max_residual, abs(energy - power * h))
            assert segment["start"] >= arrivals[sid].ceil("15min")
            assert segment["end"] <= departures[sid]
            times, targets = history[sid]
            index = bisect_right(times, segment["start"].value) - 1
            assert index >= 0, f"Unreleased request: {sid}"
            assert segment["start"] >= pd.Timestamp(
                times[0], tz="UTC").ceil("15min")
            before = cumulative[sid]
            cumulative[sid] += energy
            # Demand reductions may leave irreversible excess, but no new excess.
            assert cumulative[sid] <= max(before, targets[index]) + 1e-5

    check(prefix + ": connection and independent information audit", True)
    check(prefix + ": segment energy conservation", max_residual <= 1e-5)
    check(prefix + ": station and device limits",
          capacity_residual <= 1e-6 and device_residual <= 1e-6)
    check(prefix + ": cumulative session energy audit",
          all(abs(cumulative[s["id"]] - s["delivered_kwh"]) <= 1e-5
              for s in e.completed))
    if executor is not None:
        check(prefix + ": actual ACN-Sim cumulative feedback",
              all(abs(cumulative[sid] - ev.energy_delivered) <= 1e-5
                  for sid, ev in executor.evs.items()) and
              not executor._connected)
    return cumulative, max_residual

summaries = {}
for cohort in ["main_at_arrival", "supplement_after_submission"]:
    selected = sessions[
        (sessions["split"] == "development") &
        (sessions["cohort"] == cohort) &
        (sessions["connectionTime"] >= start) &
        (sessions["connectionTime"] < arrival_end)
    ].copy()
    check(cohort + ": nonempty fixed first-week selection", len(selected) > 0)
    ids = set(selected["sessionID"])
    selected_events = events[events["sessionID"].isin(ids)].copy()
    selected_releases = releases[releases["sessionID"].isin(ids)].copy()
    last_departure = selected_events.loc[
        selected_events["event_type"] == "departure", "event_time"].max()
    end = max(arrival_end, last_departure.ceil("15min"))
    check(cohort + ": tail remains inside development split", end <= dev_end)

    results = {}
    for backend, name in [(False, "ideal"), (True, "acnsim")]:
        e, executor = run(
            selected, selected_events, selected_releases,
            start, end, backend, OUT / cohort / name)
        totals, residual = audit(
            e, executor, selected, selected_events, f"{cohort}/{name}")
        results[name] = totals

    check(cohort + ": ideal and adapted real-data energy agree",
          all(abs(results["ideal"][sid] - results["acnsim"][sid]) <= 1e-5
              for sid in ids))
    total = sum(results["acnsim"].values())
    check(cohort + ": replay includes positive actual delivery", total > 0)
    summaries[cohort] = {
        "sessions": len(selected),
        "arrival_start_utc": str(start),
        "arrival_end_utc_exclusive": str(arrival_end),
        "execution_end_utc": str(end),
        "diagnostic_delivered_kwh": total,
    }

# Synthetic DST tests exercise actual execution over a full local natural day.
for date, expected_hours in [("2019-03-10", 23), ("2019-11-03", 25)]:
    local_start = pd.Timestamp(date, tz=TZ)
    local_end = local_start + pd.DateOffset(days=1)
    begin, end = local_start.tz_convert("UTC"), local_end.tz_convert("UTC")
    sid = "synthetic-dst"
    selected = pd.DataFrame([{
        "sessionID": sid, "stationID": "dst-evse",
        "connectionTime": begin, "cohort": "main_at_arrival",
        "initial_kWhRequested": 500.0,
        "initial_requestedDeparture": end,
    }])
    selected_events = pd.DataFrame([
        {"sessionID": sid, "event_type": "arrival", "event_time": begin},
        {"sessionID": sid, "event_type": "departure", "event_time": end},
    ])
    for backend, name in [(False, "ideal"), (True, "acnsim")]:
        e, executor = run(
            selected, selected_events, pd.DataFrame(),
            begin, end, backend, None,
            control=lambda view: {sid: 6.0} if view["vehicles"] else {})
        check(f"DST {date}/{name}: physical duration and energy",
              len(e.ledger) == expected_hours * 4 and
              abs(e.completed[0]["delivered_kwh"] - 6 * expected_hours) <= 1e-5)
        check(f"DST {date}/{name}: one local calendar day",
              all(str(row["start"].tz_convert(TZ).date()) == date
                  for row in e.ledger))

# Verify the previously supplied independent regression evidence.
previous = ROOT / "results/logs/stage3_adapter_20261112T074129922228Z"
for name, count in [
    ("event_regression", 22), ("execution_regression", 13),
    ("adapter_integration", 20),
]:
    evidence = json.loads(
        (previous / f"{name}_verification.json").read_text(encoding="utf-8"))
    check(f"prior {name}: saved evidence",
          evidence["status"] == "passed" and evidence["check_count"] == count)

tracked = [
    ROOT / "src/simulator.py", ROOT / "src/acnsim_adapter.py",
    Path(__file__), DATA / "sessions_with_split.csv",
    DATA / "environment_events.csv", DATA / "input_release_events.csv",
    DATA / "user_updates.csv", DATA / "information_policy.json",
    ROOT / "configs/station.yaml", ROOT / "configs/splits.yaml",
]
hashes = {str(p.relative_to(ROOT)): sha256(p) for p in tracked}
for filename in ["station.yaml", "splits.yaml"]:
    shutil.copy2(ROOT / "configs" / filename, OUT / filename)
shutil.copy2(DATA / "information_policy.json", OUT / "information_policy.json")

interface_note = """# Stage 3.1 environment and adapter interfaces

- AggregateEnvironment owns full events; the controller receives view() copies.
- view() contains current UTC time, capacity and already-connected vehicles.
- Visible vehicle fields: id, station_id, arrival, cohort, released target_kwh,
  expected_departure, delivered_kwh, pmax_kw, remaining_kwh,
  irreversible_excess_kwh. It contains no future disconnectTime or arrivals.
- step({session_id: kW}) executes one 15-minute control interval.
- Arrival rounds up for control eligibility; visible expected departure rounds
  down. Expired estimates use one-slot extension in the diagnostic controller.
- Same-time event order: departure, arrival, input update; last input_order wins.
- Main requests freeze at arrival. Supplementary inputs release at modifiedAt.
- The executor receives only current visible state and requested segment power.
- ACN-Sim network feasibility is checked before update_pilots.
- Actual current feedback drives grid-side delivered energy; efficiency is 1.
- Ledger and completed-session outputs belong to the evaluator, not controller.
- ACN-Sim components execute devices; the project environment advances events.
- Network validation covers the aggregate experimental capacity scenario only.
- Artificial battery reservoirs provide bookkeeping, not inferred SOC/capacity.
- Diagnostic replay is not a formal method comparison or normalization baseline.
"""
(OUT / "environment_interfaces.md").write_text(interface_note, encoding="utf-8")

result = {
    "status": "passed", "check_count": len(passed), "checks": passed,
    "development_replays": summaries, "source_and_input_sha256": hashes,
    "scope": "stage 3.1 correctness checks; no formal algorithm comparison",
    "original_caltech_network_validated": False,
}
(OUT / "verification.json").write_text(
    json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(
    {k: v for k, v in result.items() if k != "source_and_input_sha256"},
    ensure_ascii=False, indent=2))
print("Hash manifest saved in verification.json")
print("Interface note:", OUT / "environment_interfaces.md")
'''

path = ROOT / "experiments/validate_environment.py"
text = source.lstrip()
compile(text, str(path), "exec")
if path.exists() and path.read_text(encoding="utf-8") != text:
    raise FileExistsError(f"保留已有不同文件，未覆盖：{path}")
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(text, encoding="utf-8")

output = LOG / "stdout_stderr.txt"
with output.open("w", encoding="utf-8") as stream:
    try:
        result = subprocess.run(
            [sys.executable, str(path), str(LOG)],
            cwd=ROOT, stdout=stream, stderr=subprocess.STDOUT,
            text=True, timeout=300,
        )
        code = result.returncode
    except subprocess.TimeoutExpired:
        code = "TIMEOUT"

print("解释器：", sys.executable)
print("退出码：", code)
print("证据目录：", LOG)
print(output.read_text(encoding="utf-8"))

解释器： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/.envs/ev-charging/bin/python
退出码： 0
证据目录： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/results/logs/stage3_environment_acceptance_20261112T074645020514Z
/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/.envs/ev-charging/lib/python3.11/site-packages/acnportal/acnsim/base.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
{
  "status": "passed",
  "check_count": 43,
  "checks": [
    "frozen aggregate scope",
    "device power and current units agree",
    "main_at_arrival: nonempty fixed first-week selection",
    "main_at_arrival: tail remains inside development split",
    "main_at_arrival/ideal: complete selected-session replay",
    "main_at_arrival/ideal: connection and independ

In [9]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import sys

ROOT = Path("/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
LOG = ROOT / "results/logs" / f"stage3_baselines_{stamp}"
LOG.mkdir(parents=True, exist_ok=False)

baseline_source = r'''
"""Online rule baselines. No file access or environment truth is used."""
import math
import pandas as pd

EPS = 1e-9

def _number(value):
    value = float(value)
    if not math.isfinite(value) or value < 0:
        raise ValueError("Expected finite nonnegative value")
    return value

def _time(value):
    value = pd.Timestamp(value)
    if pd.isna(value) or value.tzinfo is None:
        raise ValueError("Timezone-aware timestamp required")
    return value.tz_convert("UTC")

def _state(view, interval_minutes):
    if interval_minutes <= 0 or int(interval_minutes) != interval_minutes:
        raise ValueError("Positive integer interval required")
    delta = pd.Timedelta(minutes=int(interval_minutes))
    now = _time(view["time"])
    if now.value % delta.value:
        raise ValueError("Decision time is not on the control grid")
    capacity = _number(view["capacity_kw"])
    cars, ids, stations = [], set(), set()
    for original in view["vehicles"]:
        sid = original["id"]
        station = original["station_id"]
        if sid in ids or station in stations:
            raise ValueError("Duplicate session or EVSE occupancy")
        ids.add(sid)
        stations.add(station)
        arrival = _time(original["arrival"])
        if arrival > now:
            raise ValueError("Future arrival supplied to online controller")
        if original["target_kwh"] is None:
            continue
        target = _number(original["target_kwh"])
        delivered = _number(original["delivered_kwh"])
        remaining = max(target - delivered, 0.0)
        if abs(remaining - _number(original["remaining_kwh"])) > 1e-8:
            raise ValueError("Inconsistent remaining demand")
        if remaining <= EPS or arrival.ceil(delta) > now:
            continue
        cars.append({
            **original, "arrival": arrival, "remaining_kwh": remaining,
            "pmax_kw": _number(original["pmax_kw"]),
        })
    return now, delta, capacity, cars

def uncontrolled(view, interval_minutes=15, details=False):
    """Capacity-limited immediate charging; arrival then session ID order.

    Raw requests are device maxima. Allocation also caps each first-slot
    energy by the current remaining demand.
    """
    now, delta, capacity, cars = _state(view, interval_minutes)
    dt = delta.value / 3_600_000_000_000
    command = {}
    requests = {}
    left = capacity
    remaining_after = {}
    for car in sorted(cars, key=lambda v: (v["arrival"], v["id"])):
        sid = car["id"]
        requests[sid] = car["pmax_kw"]
        power = min(car["pmax_kw"], left, car["remaining_kwh"] / dt)
        if power > 0:
            command[sid] = power
            left = max(left - power, 0.0)
        remaining_after[sid] = max(car["remaining_kwh"] - power * dt, 0.0)
    if details:
        return {
            "command": command, "raw_request_kw": requests,
            "slot": now, "remaining_after_first_slot_kwh": remaining_after,
        }
    return command

def tou_greedy(view, price_at, interval_minutes=15,
               base_hours=24.0, details=False):
    """Rebuild a price-sorted plan for currently visible vehicles only.

    Vehicles: expected departure, arrival, ID.
    Slots: price, then earlier time.
    Expired estimates extend by one slot. Future arrivals are not predicted.
    price_at must represent a preannounced tariff calendar.
    """
    now, delta, capacity, cars = _state(view, interval_minutes)
    if not math.isfinite(float(base_hours)) or base_hours <= 0:
        raise ValueError("Positive finite base horizon required")
    dt = delta.value / 3_600_000_000_000

    deadlines = {}
    for car in cars:
        estimate = _time(car["expected_departure"])
        car["expected_departure"] = estimate
        deadlines[car["id"]] = (
            now + delta if estimate <= now else estimate.floor(delta))

    if not cars:
        result = {
            "command": {}, "slots": (), "plan_kw": {},
            "forecast_unmet_kwh": {}, "effective_horizon_hours": 0.0,
        }
        return result if details else {}

    end = max(
        (now + pd.Timedelta(hours=base_hours)).ceil(delta),
        max(deadlines.values()))
    slots = tuple(pd.date_range(now, end, freq=delta, inclusive="left"))
    prices = [_number(price_at(t)) for t in slots]
    capacity_left = [capacity] * len(slots)
    plan, unmet = {}, {}

    for car in sorted(
        cars, key=lambda v: (v["expected_departure"], v["arrival"], v["id"])
    ):
        sid = car["id"]
        powers = [0.0] * len(slots)
        remaining = car["remaining_kwh"]
        available = [
            j for j, t in enumerate(slots)
            if t >= car["arrival"].ceil(delta) and
            t + delta <= deadlines[sid]
        ]
        for j in sorted(available, key=lambda j: (prices[j], slots[j])):
            power = min(car["pmax_kw"], capacity_left[j], remaining / dt)
            powers[j] = power
            capacity_left[j] = max(capacity_left[j] - power, 0.0)
            remaining = max(remaining - power * dt, 0.0)
            if remaining <= EPS:
                break
        plan[sid] = tuple(powers)
        unmet[sid] = remaining

    command = {sid: powers[0] for sid, powers in plan.items() if powers[0] > 0}
    result = {
        "command": command, "slots": slots, "plan_kw": plan,
        "forecast_unmet_kwh": unmet,
        "effective_horizon_hours": (end - now).value / 3_600_000_000_000,
    }
    return result if details else command
'''

test_source = r'''
from pathlib import Path
from copy import deepcopy
import json
import sys
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
from src.baselines import uncontrolled, tou_greedy
from src.simulator import AggregateEnvironment
from src.acnsim_adapter import ACNSimExecutor

T = pd.Timestamp("2019-05-06T07:00:00Z")
passed = []

def check(name, condition):
    if not condition:
        raise AssertionError(name)
    passed.append(name)

def car(sid, demand=1.5, deadline=60, arrival=None):
    return {
        "id": sid, "station_id": sid,
        "arrival": T if arrival is None else arrival,
        "target_kwh": demand, "delivered_kwh": 0.0,
        "remaining_kwh": demand, "pmax_kw": 6.0,
        "expected_departure": T + pd.Timedelta(minutes=deadline),
    }

def view(cars=(), capacity=6.0):
    return {"time": T, "capacity_kw": capacity, "vehicles": tuple(cars)}

check("empty state", uncontrolled(view()) == {} and
      tou_greedy(view(), lambda t: 1) == {})

v = view([car("b"), car("a")])
check("Uncontrolled deterministic ID tie-break",
      uncontrolled(v) == {"a": 6.0})
older = car("b", arrival=T - pd.Timedelta(minutes=15))
check("Uncontrolled arrival order precedes ID",
      uncontrolled(view([car("a"), older])) == {"b": 6.0})

check("remaining energy caps first-slot power",
      uncontrolled(view([car("a", demand=0.25)])) == {"a": 1.0})
unknown = car("unknown")
unknown.update(target_kwh=None, remaining_kwh=None)
check("unreleased request skipped",
      uncontrolled(view([unknown])) == {} and
      tou_greedy(view([unknown]), lambda t: 1) == {})

before = deepcopy(v)
uncontrolled(v)
tou_greedy(v, lambda t: 1)
check("controllers do not mutate visible state", v == before)

cheap_later = lambda t: 1.0 if t == T else 0.1
check("TOU waits for cheaper available slot",
      tou_greedy(view([car("a")]), cheap_later, base_hours=1) == {})
check("TOU equal prices prefer earlier slot",
      tou_greedy(view([car("a")]), lambda t: 1, base_hours=1) == {"a": 6.0})
check("TOU respects imminent expected departure",
      tou_greedy(view([car("a", deadline=15)]),
                 cheap_later, base_hours=1) == {"a": 6.0})
check("TOU prioritizes earlier-departing vehicle",
      tou_greedy(view([car("a", deadline=30), car("b", deadline=15)]),
                 lambda t: 1, base_hours=1) == {"b": 6.0})
check("TOU expired estimate extends by one slot",
      tou_greedy(view([car("a", deadline=-15)]),
                 lambda t: 1, base_hours=1) == {"a": 6.0})
check("TOU future partial slot is unavailable",
      tou_greedy(view([car("a", deadline=10)]),
                 lambda t: 1, base_hours=1) == {})

extended = tou_greedy(
    view([car("a", deadline=30 * 60)]),
    lambda t: 0.1 if t >= T + pd.Timedelta(hours=29) else 1,
    details=True)
check("TOU extends horizon using visible estimate",
      extended["effective_horizon_hours"] == 30 and
      extended["command"] == {} and
      abs(sum(extended["plan_kw"]["a"]) * 0.25 - 1.5) < 1e-8)

r = tou_greedy(
    view([car("a", demand=10, deadline=30),
          car("b", demand=10, deadline=30)]),
    lambda t: 1, base_hours=1, details=True)
check("TOU all future slots respect shared capacity",
      all(sum(powers[j] for powers in r["plan_kw"].values()) <= 6 + 1e-8
          for j in range(len(r["slots"]))))
check("TOU keeps infeasible residual demand",
      abs(sum(r["forecast_unmet_kwh"].values()) - 17.0) < 1e-8)
check("TOU planned energy and residual conserve each demand",
      all(abs(sum(r["plan_kw"][sid]) * 0.25 +
                  r["forecast_unmet_kwh"][sid] - 10) < 1e-8
          for sid in ["a", "b"]))

reversed_v = view(list(reversed(v["vehicles"])))
check("TOU deterministic under input permutation",
      tou_greedy(v, lambda t: 1, details=True) ==
      tou_greedy(reversed_v, lambda t: 1, details=True))

# Run both controllers through the accepted ACN-Sim execution environment.
for method in ["Uncontrolled", "TOU Greedy"]:
    sessions, events = [], []
    for sid in ["a", "b"]:
        sessions.append({
            "sessionID": sid, "stationID": sid,
            "connectionTime": T, "cohort": "main_at_arrival",
            "initial_kWhRequested": 1.5,
            "initial_requestedDeparture": T + pd.Timedelta(minutes=30),
        })
        for kind, minute in [("arrival", 0), ("departure", 60)]:
            events.append({
                "sessionID": sid, "event_type": kind,
                "event_time": T + pd.Timedelta(minutes=minute),
            })
    executor = ACNSimExecutor(["a", "b"], capacity_kw=6)
    e = AggregateEnvironment(
        pd.DataFrame(sessions), pd.DataFrame(events), pd.DataFrame(),
        start=T, capacity_kw=6, executor=executor)
    for _ in range(4):
        state = e.view()
        command = (
            uncontrolled(state) if method == "Uncontrolled" else
            tou_greedy(state, lambda t: 0.12))
        e.step(command)
    check(method + ": ACN-Sim integrated delivery",
          len(e.completed) == 2 and
          all(abs(s["delivered_kwh"] - 1.5) < 1e-8 for s in e.completed) and
          all(row["total_kw"] <= 6 + 1e-8 for row in e.ledger))

result = {
    "status": "passed", "check_count": len(passed), "checks": passed,
    "scope": "baseline synthetic checks; real development replay still pending",
}
Path(sys.argv[1]).write_text(
    json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(result, ensure_ascii=False, indent=2))
'''

files = {
    ROOT / "src/baselines.py": baseline_source.lstrip(),
    ROOT / "tests/test_baselines.py": test_source.lstrip(),
}
for path, text in files.items():
    compile(text, str(path), "exec")
    if path.exists() and path.read_text(encoding="utf-8") != text:
        raise FileExistsError(f"保留已有不同文件，未覆盖：{path}")
for path, text in files.items():
    path.write_text(text, encoding="utf-8")

output = LOG / "stdout_stderr.txt"
with output.open("w", encoding="utf-8") as stream:
    try:
        result = subprocess.run(
            [sys.executable, str(ROOT / "tests/test_baselines.py"),
             str(LOG / "verification.json")],
            cwd=ROOT, stdout=stream, stderr=subprocess.STDOUT,
            text=True, timeout=180,
        )
        code = result.returncode
    except subprocess.TimeoutExpired:
        code = "TIMEOUT"

print("解释器：", sys.executable)
print("退出码：", code)
print("证据目录：", LOG)
print(output.read_text(encoding="utf-8"))

解释器： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/.envs/ev-charging/bin/python
退出码： 0
证据目录： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/results/logs/stage3_baselines_20261112T075222486951Z
/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/.envs/ev-charging/lib/python3.11/site-packages/acnportal/acnsim/base.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
{
  "status": "passed",
  "check_count": 19,
  "checks": [
    "empty state",
    "Uncontrolled deterministic ID tie-break",
    "Uncontrolled arrival order precedes ID",
    "remaining energy caps first-slot power",
    "unreleased request skipped",
    "controllers do not mutate visible state",
    "TOU waits for cheaper available slot",
    "TOU equal prices prefer earlier slo

In [10]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import sys

ROOT = Path("/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
LOG = ROOT / "results/logs" / f"stage3_baseline_replay_{stamp}"
LOG.mkdir(parents=True, exist_ok=False)

source = r'''
from pathlib import Path
from bisect import bisect_right
from statistics import median
import hashlib
import json
import math
import sys
import pandas as pd
import yaml

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
from src.baselines import uncontrolled, tou_greedy
from src.simulator import AggregateEnvironment
from src.acnsim_adapter import ACNSimExecutor

OUT = Path(sys.argv[1])
DATA = ROOT / "data/processed/caltech_12weeks_20261111T143402Z_9b087d"
passed = []

def check(name, condition):
    if not condition:
        raise AssertionError(name)
    passed.append(name)

def load_config(name):
    return yaml.safe_load(
        (ROOT / "configs" / name).read_text(encoding="utf-8"))

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

station = load_config("station.yaml")
tariff = load_config("tariff.yaml")
splits = load_config("splits.yaml")
TZ = station["timezone"]
start = pd.Timestamp(splits["splits"]["development"]["start_utc_inclusive"])
end = pd.Timestamp(splits["splits"]["development"]["end_utc_exclusive"])
capacity = float(station["station_capacity_kW"])
pmax = float(station["evse_max_power_kW"])

check("frozen execution scope",
      station["network_model"] == "aggregate_continuous" and
      station["control_interval_minutes"] == 15 and
      station["charging_efficiency"] == 1 and
      not station["original_branch_constraints_enforced"])
check("tariff scope",
      tariff["timezone"] == TZ and tariff["unit"] == "USD/kWh" and
      tariff["calendar"] == "same_schedule_every_day_including_weekends_and_holidays" and
      tariff["extension_policy"] == "repeat_local_daily_schedule_for_any_date")

def minute(value):
    if isinstance(value, int):
        return value
    hour, mins = map(int, str(value).split(":"))
    return hour * 60 + mins

periods = sorted(
    (minute(p["start"]), minute(p["end"]), float(p["price"]))
    for p in tariff["periods"])
check("tariff continuous and nonnegative",
      periods[0][0] == 0 and periods[-1][1] == 1440 and
      all(a < b and math.isfinite(price) and price >= 0
          for a, b, price in periods) and
      all(periods[j][1] == periods[j + 1][0]
          for j in range(len(periods) - 1)))

price_cache = {}
def price_at(t):
    t = pd.Timestamp(t)
    key = t.value
    if key not in price_cache:
        local = t.tz_convert(TZ)
        m = local.hour * 60 + local.minute
        price_cache[key] = next(price for a, b, price in periods if a <= m < b)
    return price_cache[key]

for hour, expected in [(0, .12), (8, .24), (16, .40), (21, .24)]:
    t = pd.Timestamp(f"2019-05-06 {hour:02}:00", tz=TZ)
    check(f"tariff boundary {hour:02}:00", abs(price_at(t) - expected) < 1e-12)

sessions = pd.read_csv(DATA / "sessions_with_split.csv")
sessions = sessions[sessions["split"] == "development"].copy()
sessions["connectionTime"] = pd.to_datetime(sessions["connectionTime"], utc=True)
events = pd.read_csv(DATA / "environment_events.csv")
events["event_time"] = pd.to_datetime(events["event_time"], utc=True)
releases = pd.read_csv(DATA / "input_release_events.csv")
updates = pd.read_csv(DATA / "user_updates.csv")

def valid(v):
    return str(v).strip().lower() in {"true", "1", "1.0"}

dates = [str(t.date()) for t in pd.date_range(
    start.tz_convert(TZ), end.tz_convert(TZ), freq="D", inclusive="left")]
check("49 complete local development days", len(dates) == 49)

def histories(selected, selected_events):
    arrivals = dict(zip(selected["sessionID"], selected["connectionTime"]))
    departures = dict(selected_events.loc[
        selected_events["event_type"] == "departure",
        ["sessionID", "event_time"]].itertuples(index=False, name=None))
    history = {}
    for row in selected.to_dict("records"):
        sid = row["sessionID"]
        if row["cohort"] == "main_at_arrival":
            times = [arrivals[sid].value]
            targets = [float(row["initial_kWhRequested"])]
        else:
            u = updates[updates["sessionID"] == sid].copy()
            u = u[u["valid"].map(valid)]
            u["modifiedAt"] = pd.to_datetime(u["modifiedAt"], utc=True)
            u = u[(u["modifiedAt"] >= arrivals[sid]) &
                  (u["modifiedAt"] < departures[sid])]
            u = u.sort_values(["modifiedAt", "input_order"])
            times = [t.value for t in u["modifiedAt"]]
            targets = u["kWhRequested"].astype(float).tolist()
        assert times, f"No independently valid target: {sid}"
        history[sid] = (times, targets)
    return arrivals, departures, history

summaries, daily_tables = [], {}
for cohort, count in [
    ("main_at_arrival", 64),
    ("supplement_after_submission", 1227),
]:
    selected = sessions[sessions["cohort"] == cohort].copy()
    check(cohort + ": development sample count", len(selected) == count)
    ids = set(selected["sessionID"])
    selected_events = events[events["sessionID"].isin(ids)].copy()
    selected_releases = releases[releases["sessionID"].isin(ids)].copy()
    arrivals, departures, history = histories(selected, selected_events)
    check(cohort + ": complete split boundaries",
          all(start <= arrivals[sid] < departures[sid] <= end for sid in ids))

    for method in ["Uncontrolled", "TOU_Greedy"]:
        print(f"Running {cohort}/{method}", flush=True)
        folder = OUT / cohort / method
        folder.mkdir(parents=True)
        executor = ACNSimExecutor(
            selected["stationID"].astype(str).tolist(),
            capacity_kw=capacity, voltage=float(station["evse_voltage_V"]),
            max_current=float(station["evse_max_current_A"]))
        e = AggregateEnvironment(
            selected, selected_events, selected_releases, start=start,
            capacity_kw=capacity, pmax_kw=pmax, executor=executor)

        with (folder / "requests_and_plans.jsonl").open("w", encoding="utf-8") as stream:
            while e.now < end:
                state = e.view()
                if method == "Uncontrolled":
                    result = uncontrolled(state, details=True)
                    record = {"time": str(e.now), **result}
                else:
                    result = tou_greedy(state, price_at, details=True)
                    # Sparse encoding preserves the entire future plan.
                    record = {
                        "time": str(e.now), "command": result["command"],
                        "slot_count": len(result["slots"]),
                        "effective_horizon_hours": result["effective_horizon_hours"],
                        "forecast_unmet_kwh": result["forecast_unmet_kwh"],
                        "sparse_plan_kw": {
                            sid: [[j, power] for j, power in enumerate(powers) if power > 0]
                            for sid, powers in result["plan_kw"].items()
                        },
                    }
                stream.write(json.dumps(record, default=str) + "\n")
                e.step(result["command"])

        prefix = f"{cohort}/{method}"
        check(prefix + ": all selected sessions completed",
              len(e.completed) == count and not e.view()["vehicles"] and
              not executor._connected)

        cumulative = {sid: 0.0 for sid in ids}
        vehicle_cost = {sid: 0.0 for sid in ids}
        daily = {
            date: {"local_date": date, "energy_kwh": 0.0,
                   "cost_USD": 0.0, "peak_kW": 0.0}
            for date in dates
        }
        station_rows, energy_rows = [], []
        energy_residual = power_residual = 0.0

        for segment in e.ledger:
            a, b = segment["start"], segment["end"]
            h = (b - a).value / 3_600_000_000_000
            assert h > 0
            date = str(a.tz_convert(TZ).date())
            assert date == str((b - pd.Timedelta(nanoseconds=1)).tz_convert(TZ).date())
            assert price_at(a) == price_at(b - pd.Timedelta(nanoseconds=1))
            rate = price_at(a)
            energy = sum(segment["energy_kwh"].values())
            energy_residual = max(
                energy_residual, abs(energy - segment["total_kw"] * h))
            power_residual = max(
                power_residual, segment["total_kw"] - capacity)
            daily[date]["energy_kwh"] += energy
            daily[date]["cost_USD"] += energy * rate
            daily[date]["peak_kW"] = max(
                daily[date]["peak_kW"], segment["total_kw"])
            station_rows.append({
                "start": a, "end": b, "total_kw": segment["total_kw"],
                "energy_kwh": energy, "price_USD_per_kWh": rate,
            })

            for sid, amount in segment["energy_kwh"].items():
                power = segment["power_kw"][sid]
                power_residual = max(power_residual, power - pmax, -power)
                energy_residual = max(energy_residual, abs(amount - power * h))
                assert a >= arrivals[sid].ceil("15min") and b <= departures[sid]
                times, targets = history[sid]
                j = bisect_right(times, a.value) - 1
                assert j >= 0
                assert a >= pd.Timestamp(times[0], tz="UTC").ceil("15min")
                before = cumulative[sid]
                cumulative[sid] += amount
                assert cumulative[sid] <= max(before, targets[j]) + 1e-5
                vehicle_cost[sid] += amount * rate
                energy_rows.append({
                    "sessionID": sid, "start": a, "end": b,
                    "actual_kw": power, "energy_kwh": amount,
                })

        check(prefix + ": independent connection and information audit", True)
        check(prefix + ": power and energy residuals",
              power_residual <= 1e-6 and energy_residual <= 1e-5)
        check(prefix + ": actual cumulative feedback",
              all(abs(cumulative[sid] - executor.evs[sid].energy_delivered) <= 1e-5
                  for sid in ids) and
              all(abs(cumulative[s["id"]] - s["delivered_kwh"]) <= 1e-5
                  for s in e.completed))

        daily_frame = pd.DataFrame(daily.values())
        cost = float(daily_frame["cost_USD"].sum())
        check(prefix + ": independent per-session cost reconciliation",
              abs(cost - sum(vehicle_cost.values())) <= 1e-5)
        check(prefix + ": all 49 days retained", len(daily_frame) == 49)

        service_rows = []
        for sid in sorted(ids):
            target = history[sid][1][-1]
            delivered = cumulative[sid]
            service_rows.append({
                "sessionID": sid, "evaluation_target_kwh": target,
                "delivered_kwh": delivered,
                "unmet_kwh": max(target - delivered, 0),
                "irreversible_excess_kwh": max(delivered - target, 0),
                "actual_departure": departures[sid], "cost_USD": vehicle_cost[sid],
            })
        services = pd.DataFrame(service_rows)
        total_target = float(services["evaluation_target_kwh"].sum())
        satisfaction = sum(
            min(r["delivered_kwh"], r["evaluation_target_kwh"])
            for r in service_rows) / total_target if total_target > 0 else None

        pd.DataFrame(station_rows).to_csv(folder / "station_trajectory.csv", index=False)
        pd.DataFrame(energy_rows).to_csv(folder / "session_trajectory.csv", index=False)
        services.to_csv(folder / "session_service.csv", index=False)
        daily_frame.to_csv(folder / "daily_metrics.csv", index=False)
        daily_tables[(cohort, method)] = daily_frame
        summaries.append({
            "cohort": cohort, "method": method, "sessions": count,
            "cost_USD": cost,
            "max_peak_kW": float(daily_frame["peak_kW"].max()),
            "delivered_kwh": float(services["delivered_kwh"].sum()),
            "unmet_kwh": float(services["unmet_kwh"].sum()),
            "energy_satisfaction": satisfaction,
            "uncompleted_sessions": int((services["unmet_kwh"] > .01).sum()),
            "max_power_residual_kW": power_residual,
            "max_energy_residual_kwh": energy_residual,
        })

# Prespecified normalization: all days, then positive-day fallback, then unit fallback.
reference = daily_tables[("main_at_arrival", "Uncontrolled")]
def reference_value(values):
    values = [float(v) for v in values]
    value = float(median(values))
    positive = [v for v in values if v > 0]
    if value > 0:
        rule = "all_day_median"
    elif positive:
        value, rule = float(median(positive)), "positive_day_median"
    else:
        value, rule = 1.0, "unit_fallback"
    return value, {
        "rule": rule, "zero_days": sum(v == 0 for v in values),
        "positive_days": len(positive),
    }

C_ref, cost_rule = reference_value(reference["cost_USD"])
P_ref, peak_rule = reference_value(reference["peak_kW"])
check("positive finite normalization references",
      math.isfinite(C_ref) and C_ref > 0 and math.isfinite(P_ref) and P_ref > 0)

tracked = [
    ROOT / "src/baselines.py", ROOT / "src/simulator.py",
    ROOT / "src/acnsim_adapter.py", Path(__file__),
    ROOT / "configs/station.yaml", ROOT / "configs/tariff.yaml",
    ROOT / "configs/splits.yaml", DATA / "sessions_with_split.csv",
    DATA / "environment_events.csv", DATA / "input_release_events.csv",
    DATA / "user_updates.csv",
]
hashes = {str(p.relative_to(ROOT)): sha256(p) for p in tracked}
normalization = {
    "C_ref": C_ref, "P_ref": P_ref,
    "C_ref_unit": "USD", "P_ref_unit": "kW",
    "source_split": "development", "source_cohort": "main_at_arrival",
    "source_method": "capacity_limited_uncontrolled",
    "natural_days_including_zero_days": 49,
    "cost_reference_rule": cost_rule, "peak_reference_rule": peak_rule,
    "fixed_for_all_methods_weights_cohorts_and_test_days": True,
    "evidence_directory": str(OUT.relative_to(ROOT)),
    "source_and_input_sha256": hashes,
}
proposed = yaml.safe_dump(normalization, allow_unicode=True, sort_keys=False)
(OUT / "normalization_proposed.yaml").write_text(proposed, encoding="utf-8")

frozen_path = ROOT / "configs/normalization.yaml"
if frozen_path.exists():
    existing = yaml.safe_load(frozen_path.read_text(encoding="utf-8"))
    check("existing frozen references agree",
          abs(existing["C_ref"] - C_ref) <= 1e-12 and
          abs(existing["P_ref"] - P_ref) <= 1e-12 and
          existing["source_and_input_sha256"] == hashes)
else:
    # Open exclusively: existing frozen settings cannot be silently overwritten.
    with frozen_path.open("x", encoding="utf-8") as stream:
        stream.write(proposed)

pd.DataFrame(summaries).to_csv(OUT / "development_summary.csv", index=False)
result = {
    "status": "passed", "check_count": len(passed), "checks": passed,
    "development_summary": summaries,
    "normalization": {
        "C_ref": C_ref, "P_ref": P_ref,
        "cost_reference_rule": cost_rule, "peak_reference_rule": peak_rule,
        "file": str(frozen_path),
    },
    "scope": "development baseline validation; not the formal test comparison",
}
(OUT / "verification.json").write_text(
    json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(result, ensure_ascii=False, indent=2))
'''

path = ROOT / "experiments/validate_baselines.py"
text = source.lstrip()
compile(text, str(path), "exec")
if path.exists() and path.read_text(encoding="utf-8") != text:
    raise FileExistsError(f"保留已有不同文件，未覆盖：{path}")
path.write_text(text, encoding="utf-8")

output = LOG / "stdout_stderr.txt"
with output.open("w", encoding="utf-8") as stream:
    try:
        result = subprocess.run(
            [sys.executable, str(path), str(LOG)],
            cwd=ROOT, stdout=stream, stderr=subprocess.STDOUT,
            text=True, timeout=600,
        )
        code = result.returncode
    except subprocess.TimeoutExpired:
        code = "TIMEOUT"

print("解释器：", sys.executable)
print("退出码：", code)
print("证据目录：", LOG)
print(output.read_text(encoding="utf-8"))

解释器： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/.envs/ev-charging/bin/python
退出码： 1
证据目录： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/results/logs/stage3_baseline_replay_20261112T075745118098Z
/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/.envs/ev-charging/lib/python3.11/site-packages/acnportal/acnsim/base.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Running main_at_arrival/Uncontrolled
Running main_at_arrival/TOU_Greedy
Running supplement_after_submission/Uncontrolled
Traceback (most recent call last):
  File "timedeltas.pyx", line 1807, in pandas._libs.tslibs.timedeltas.Timedelta.__new__
OverflowError: int too big to convert

The above exception was the direct cause of the following exception:

Traceback (most recent 

In [11]:
from pathlib import Path
from datetime import datetime, timezone
import json
import shutil
import subprocess
import sys

ROOT = Path("/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
LOG = ROOT / "results/logs" / f"stage3_completion_fix_{stamp}"
LOG.mkdir(parents=True, exist_ok=False)

sim_path = ROOT / "src/simulator.py"
original = sim_path.read_text(encoding="utf-8")

old = """                    until_full = max(pd.Timedelta(hours=remaining / power),
                                     pd.Timedelta(nanoseconds=1))
                    cut = min(cut, self.now + until_full)"""

new = """                    # Construct completion time only inside this event interval.
                    interval_hours = (
                        (boundary - self.now).value / 3_600_000_000_000)
                    if remaining < power * interval_hours:
                        until_full = max(
                            pd.Timedelta(hours=remaining / power),
                            pd.Timedelta(nanoseconds=1))
                        cut = min(cut, self.now + until_full)"""

if original.count(old) == 1:
    revised = original.replace(old, new)
elif new in original:
    revised = original
else:
    raise RuntimeError("源码与预期不同，停止修改；请回传此输出。")

test_source = r'''
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
from src.simulator import AggregateEnvironment
from src.acnsim_adapter import ACNSimExecutor

T = pd.Timestamp("2019-05-06T07:00:00Z")
passed = []

def check(name, condition):
    if not condition:
        raise AssertionError(name)
    passed.append(name)

def run(target, power, departure=60, adapted=False):
    sessions = pd.DataFrame([{
        "sessionID": "a", "stationID": "A",
        "connectionTime": T, "cohort": "main_at_arrival",
        "initial_kWhRequested": target,
        "initial_requestedDeparture": T + pd.Timedelta(hours=1),
    }])
    events = pd.DataFrame([
        {"sessionID": "a", "event_type": "arrival", "event_time": T},
        {"sessionID": "a", "event_type": "departure",
         "event_time": T + pd.Timedelta(minutes=departure)},
    ])
    executor = ACNSimExecutor(["A"], capacity_kw=50) if adapted else None
    e = AggregateEnvironment(
        sessions, events, pd.DataFrame(), start=T, executor=executor)
    e.step({"a": power})
    states = list(e.view()["vehicles"]) + e.completed
    return e, executor, states[0]["delivered_kwh"]

e, _, energy = run(10, 1e-20)
check("tiny positive command avoids duration overflow",
      e.now == T + pd.Timedelta(minutes=15) and
      abs(energy - 2.5e-21) < 1e-25)

# Artificial stress case only; this does not change or filter real requests.
e, _, energy = run(1e12, 6)
check("distant completion never needs a huge Timedelta",
      abs(energy - 1.5) < 1e-8)

e, _, energy = run(10, 6)
check("ordinary noncompletion energy unchanged", abs(energy - 1.5) < 1e-8)

e, _, energy = run(0.5, 6)
check("completion inside interval still stops exactly",
      abs(energy - 0.5) < 1e-8)

e, _, energy = run(1.5, 6)
check("completion exactly at interval end remains correct",
      abs(energy - 1.5) < 1e-8)

e, _, energy = run(10, 1e-20, departure=7.5)
check("tiny command still stops at actual departure",
      not e.view()["vehicles"] and abs(energy - 1.25e-21) < 1e-25)

e, executor, energy = run(10, 1e-20, adapted=True)
check("tiny command conserves ACN-Sim actual feedback",
      abs(energy - 2.5e-21) < 1e-25 and
      abs(energy - executor.evs["a"].energy_delivered) < 1e-25)

result = {"status": "passed", "check_count": len(passed), "checks": passed}
Path(sys.argv[1]).write_text(
    json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(result, ensure_ascii=False, indent=2))
'''

test_path = ROOT / "tests/test_completion_interval.py"
test_text = test_source.lstrip()
compile(revised, str(sim_path), "exec")
compile(test_text, str(test_path), "exec")
if test_path.exists() and test_path.read_text(encoding="utf-8") != test_text:
    raise FileExistsError(f"保留已有不同文件，未覆盖：{test_path}")

shutil.copy2(sim_path, LOG / "simulator_before.py")
sim_path.write_text(revised, encoding="utf-8")
test_path.write_text(test_text, encoding="utf-8")

print("解释器：", sys.executable)
print("证据目录：", LOG)

jobs = [
    ("completion_regression", test_path, 180),
    ("event_regression", ROOT / "tests/test_event_environment.py", 180),
    ("execution_regression", ROOT / "tests/test_execution_and_acnsim.py", 180),
    ("adapter_regression", ROOT / "tests/test_acnsim_adapter.py", 180),
    ("baseline_regression", ROOT / "tests/test_baselines.py", 180),
    ("baseline_replay", ROOT / "experiments/validate_baselines.py", 600),
]

for name, script, timeout in jobs:
    if name == "baseline_replay":
        destination = LOG / name
        destination.mkdir()
        verification = destination / "verification.json"
    else:
        verification = LOG / f"{name}_verification.json"
        destination = verification

    output = LOG / f"{name}_stdout_stderr.txt"
    with output.open("w", encoding="utf-8") as stream:
        try:
            result = subprocess.run(
                [sys.executable, str(script), str(destination)],
                cwd=ROOT, stdout=stream, stderr=subprocess.STDOUT,
                text=True, timeout=timeout,
            )
            code = result.returncode
        except subprocess.TimeoutExpired:
            code = "TIMEOUT"

    print(f"\n=== {name}：退出码 {code} ===")
    if code != 0 or not verification.is_file():
        print(output.read_text(encoding="utf-8"))
        print("本轮停止；后续任务未执行，全部日志已保留。")
        break

    result = json.loads(verification.read_text(encoding="utf-8"))
    print(json.dumps(result, ensure_ascii=False, indent=2))
    if result.get("status") != "passed":
        print("核验未通过，本轮停止。")
        break

解释器： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/.envs/ev-charging/bin/python
证据目录： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/results/logs/stage3_completion_fix_20261112T080048207614Z

=== completion_regression：退出码 0 ===
{
  "status": "passed",
  "check_count": 7,
  "checks": [
    "tiny positive command avoids duration overflow",
    "distant completion never needs a huge Timedelta",
    "ordinary noncompletion energy unchanged",
    "completion inside interval still stops exactly",
    "completion exactly at interval end remains correct",
    "tiny command still stops at actual departure",
    "tiny command conserves ACN-Sim actual feedback"
  ]
}

=== event_regression：退出码 0 ===
{
  "status": "passed",
  "checks": [
    "15-minute energy integral",
    "mid-slot departure stops power",
    "stop at target without overcharging",
    "late input initially hidden",
    "no charging before released request",
    "late input released without retroactive ene

In [12]:
from pathlib import Path

ROOT = Path("/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization")
print("RH_LP_INTERFACE_SNAPSHOT")

for relative in [
    "src/optimization_model.py",
    "experiments/run_static.py",
    "src/tariff.py",
    "configs/normalization.yaml",
]:
    path = ROOT / relative
    print(f"\n===== BEGIN {relative} =====")
    if path.is_file():
        print(path.read_text(encoding="utf-8"))
    else:
        print("文件缺失")
    print(f"===== END {relative} =====")

print("\nRH_LP_INTERFACE_SNAPSHOT_END")

RH_LP_INTERFACE_SNAPSHOT

===== BEGIN src/optimization_model.py =====
"""聚合连续功率两层 LP。仅接收显式输入，不读取未来事件文件。"""
from pathlib import Path
from time import perf_counter
from contextlib import redirect_stdout, redirect_stderr
import csv
import json
import math
import traceback
import pyomo.environ as pyo


def solve_lp(case, output_dir):
    """
    slots: [{label, day, price, capacity}]
    sessions: [{id, demand, pmax, available: [slot_index, ...]}]
    参数: dt, alpha, C_ref, P_ref, epsilon_U, time_limit_seconds
    available 由调用方构造；核心不判断真实/预计离站。
    """
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=False)
    (out / "input.json").write_text(
        json.dumps(case, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    started = perf_counter()
    report = {"status": "running", "stages": []}

    def finite_number(value, name, lower=None):
        if isinstance(value, bool):
            raise ValueError(f"{name} 不得为布尔值")
        value = float(value)
        if not math.

In [13]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import sys

ROOT = Path("/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
LOG = ROOT / "results/logs" / f"stage3_rolling_synthetic_{stamp}"
LOG.mkdir(parents=True, exist_ok=False)

controller_source = r'''
"""Rolling controller: visible inputs only; existing two-stage LP in a worker."""
from pathlib import Path
from time import perf_counter
import csv
import json
import math
import subprocess
import sys
import traceback
import pandas as pd
from .baselines import _state, _time, _number

ROOT = Path(__file__).resolve().parents[1]
DELTA = pd.Timedelta(minutes=15)

def deadlines_for(now, cars):
    result = {}
    for car in cars:
        estimate = _time(car["expected_departure"])
        result[car["id"]] = (
            now + DELTA if estimate <= now else estimate.floor(DELTA))
    return result

def edf_command(view):
    now, _, capacity, cars = _state(view, 15)
    deadlines = deadlines_for(now, cars)
    command = {}
    for car in sorted(cars, key=lambda v: (
        _time(v["expected_departure"]), v["arrival"], v["id"])):
        if deadlines[car["id"]] < now + DELTA:
            continue
        power = min(car["pmax_kw"], capacity, car["remaining_kwh"] / .25)
        if power > 0:
            command[car["id"]] = power
            capacity = max(capacity - power, 0.0)
    return command

def record_realized_peaks(peaks, segments, observed_until,
                          timezone="America/Los_Angeles"):
    """Update from already-executed segments, never predicted window peaks."""
    observed_until = _time(observed_until)
    for segment in segments:
        a, b = _time(segment["start"]), _time(segment["end"])
        if not a < b <= observed_until:
            raise ValueError("Invalid or future execution feedback")
        power = _number(segment["total_kw"])
        while a < b:
            local = a.tz_convert(timezone)
            day = str(local.date())
            peaks[day] = max(peaks.get(day, 0.0), power)
            midnight = (
                local.normalize() + pd.DateOffset(days=1)).tz_convert("UTC")
            a = min(b, midnight)

def _invoke(case, step, timeout):
    input_path = step / "case.json"
    input_path.write_text(
        json.dumps(case, ensure_ascii=False, indent=2), encoding="utf-8")
    with (step / "worker.txt").open("w", encoding="utf-8") as stream:
        process = subprocess.run(
            [sys.executable, "-m", "src.rolling_horizon", "--worker",
             str(input_path), str(step / "lp")],
            cwd=ROOT, stdout=stream, stderr=subprocess.STDOUT,
            timeout=timeout, text=True)
    if process.returncode != 0:
        raise RuntimeError(f"LP worker exit code {process.returncode}")
    report = json.loads((step / "lp/report.json").read_text(encoding="utf-8"))
    if report.get("status") != "passed":
        raise RuntimeError("LP report not passed")
    if len(report.get("stages", [])) != 2 or any(
        stage["termination"] != "optimal" for stage in report["stages"]
    ):
        raise RuntimeError("Both LP stages must be optimal")
    return report

def _first_command(case, plan_path):
    cars = {car["id"]: car for car in case["sessions"]}
    labels = {slot["label"]: j for j, slot in enumerate(case["slots"])}
    allowed = {sid: set(car["available"]) for sid, car in cars.items()}
    loads = [0.0] * len(labels)
    energies = {sid: 0.0 for sid in cars}
    first, seen = {}, set()
    with plan_path.open(newline="", encoding="utf-8") as stream:
        for row in csv.DictReader(stream):
            sid, label = row["sessionID"], row["slot"]
            if sid not in cars or label not in labels or (sid, label) in seen:
                raise ValueError("Invalid or duplicate exported plan key")
            seen.add((sid, label))
            j = labels[label]
            power = float(row["power_kW"])
            energy = float(row["energy_kWh"])
            if not math.isfinite(power) or not math.isfinite(energy):
                raise ValueError("Nonfinite exported plan")
            limit = cars[sid]["pmax"] if j in allowed[sid] else 0.0
            if power < -1e-6 or power > limit + 1e-6:
                raise ValueError("Exported device/availability violation")
            if row["day"] != case["slots"][j]["day"]:
                raise ValueError("Exported day mismatch")
            if abs(energy - power * case["dt"]) > 1e-8:
                raise ValueError("Exported energy integral mismatch")
            loads[j] += power
            energies[sid] += energy
            if j == 0:
                first[sid] = power
    if len(seen) != len(cars) * len(labels):
        raise ValueError("Incomplete exported plan")
    if any(load > slot["capacity"] + 1e-6
           for load, slot in zip(loads, case["slots"])):
        raise ValueError("Exported station capacity violation")
    if any(energies[sid] > cars[sid]["demand"] + 1e-5 for sid in cars):
        raise ValueError("Exported overcharge")

    # Retain raw files; remove only tolerance-scale errors for execution.
    command, adjustment = {}, 0.0
    capacity = case["slots"][0]["capacity"]
    for sid in sorted(cars):
        raw = first[sid]
        limit = min(cars[sid]["pmax"], cars[sid]["demand"] / case["dt"],
                    capacity) if 0 in allowed[sid] else 0.0
        power = min(max(raw, 0.0), limit)
        if abs(power - raw) > 1e-6:
            raise ValueError("First-slot correction exceeds power tolerance")
        adjustment += abs(power - raw)
        if power > 0:
            command[sid] = power
            capacity = max(capacity - power, 0.0)
    return command, adjustment

class RollingHorizonController:
    def __init__(self, output_dir, price_at, C_ref, P_ref,
                 alpha=.5, base_hours=24, epsilon_U=1e-5,
                 time_limit_seconds=30, timezone="America/Los_Angeles"):
        self.root = Path(output_dir)
        self.price_at = price_at
        self.C_ref, self.P_ref = _number(C_ref), _number(P_ref)
        self.alpha = _number(alpha)
        self.base_hours = _number(base_hours)
        self.epsilon = _number(epsilon_U)
        self.limit = _number(time_limit_seconds)
        if min(self.C_ref, self.P_ref, self.base_hours, self.limit) <= 0:
            raise ValueError("References, horizon and time budget must be positive")
        if self.alpha > 1:
            raise ValueError("alpha must be in [0, 1]")
        self.timezone = timezone
        self.counter = 0
        self.root.mkdir(parents=True, exist_ok=False)

    def build_case(self, view, past_peak=None):
        now, _, capacity, cars = _state(view, 15)
        deadlines = deadlines_for(now, cars)
        end = max(
            (now + pd.Timedelta(hours=self.base_hours)).ceil(DELTA),
            max(deadlines.values(), default=now))
        grid = tuple(pd.date_range(now, end, freq=DELTA, inclusive="left"))
        days = [str(t.tz_convert(self.timezone).date()) for t in grid]
        today = str(now.tz_convert(self.timezone).date())
        past = {}
        for day, value in (past_peak or {}).items():
            if not isinstance(day, str) or str(pd.Timestamp(day).date()) != day:
                raise ValueError("Invalid peak date")
            if day > today:
                raise ValueError("Future realized peak supplied")
            past[day] = _number(value)
        return {
            "dt": .25, "alpha": self.alpha,
            "C_ref": self.C_ref, "P_ref": self.P_ref,
            "epsilon_U": self.epsilon, "time_limit_seconds": self.limit,
            "past_peak": {day: past.get(day, 0.0) for day in dict.fromkeys(days)},
            "slots": [
                {"label": t.isoformat(), "day": day,
                 "price": _number(self.price_at(t)), "capacity": capacity}
                for t, day in zip(grid, days)
            ],
            "sessions": [
                {"id": car["id"], "demand": car["remaining_kwh"],
                 "pmax": car["pmax_kw"],
                 "available": [
                     j for j, t in enumerate(grid)
                     if t >= car["arrival"].ceil(DELTA) and
                     t + DELTA <= deadlines[car["id"]]
                 ]}
                for car in sorted(cars, key=lambda v: v["id"])
            ],
        }

    def decide(self, view, past_peak=None):
        tick = perf_counter()
        now, _, _, cars = _state(view, 15)
        self.counter += 1
        step = self.root / f"step_{self.counter:06d}"
        step.mkdir()
        result = {
            "time": now.isoformat(), "mode": "empty", "fallback": False,
            "reason": None, "command": {}, "solve_report": None,
            "first_slot_adjustment_kW": 0.0,
            "effective_horizon_hours": 0.0,
        }
        if cars:
            try:
                case = self.build_case(view, past_peak)
                result["effective_horizon_hours"] = len(case["slots"]) * .25
                remaining = self.limit - (perf_counter() - tick)
                if remaining <= 0:
                    raise TimeoutError("Decision budget consumed before solve")
                case["time_limit_seconds"] = remaining
                report = _invoke(case, step, remaining)
                if perf_counter() - tick >= self.limit:
                    raise TimeoutError("Decision budget consumed by solve")
                command, adjustment = _first_command(
                    case, step / "lp/plan.csv")
                if perf_counter() - tick >= self.limit:
                    raise TimeoutError("Decision budget consumed by plan validation")
                result.update(
                    mode="lp", command=command, solve_report=report,
                    first_slot_adjustment_kW=adjustment)
            except Exception as exc:
                (step / "controller_error.txt").write_text(
                    traceback.format_exc(), encoding="utf-8")
                result.update(
                    mode="fallback", fallback=True, reason=repr(exc),
                    command=edf_command(view))
        result["total_seconds"] = perf_counter() - tick
        result["step_directory"] = str(step)
        (step / "decision.json").write_text(
            json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
        return result

if __name__ == "__main__":
    if len(sys.argv) != 4 or sys.argv[1] != "--worker":
        raise SystemExit("Usage: --worker CASE_JSON OUTPUT_DIR")
    from .optimization_model import solve_lp
    solve_lp(
        json.loads(Path(sys.argv[2]).read_text(encoding="utf-8")),
        Path(sys.argv[3]))
'''

test_source = r'''
from pathlib import Path
from unittest.mock import patch
import json
import sys
import pandas as pd
import yaml

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
from src.rolling_horizon import (
    RollingHorizonController, record_realized_peaks, edf_command)
from src.simulator import AggregateEnvironment
from src.acnsim_adapter import ACNSimExecutor

OUT = Path(sys.argv[1]).parent / "rolling_cases"
OUT.mkdir()
NORM = yaml.safe_load(
    (ROOT / "configs/normalization.yaml").read_text(encoding="utf-8"))
T = pd.Timestamp("2019-05-06T07:00:00Z")
passed, controllers = [], []

def check(name, condition):
    if not condition:
        raise AssertionError(name)
    passed.append(name)

def car(sid="a", demand=1.5, deadline=30):
    return {
        "id": sid, "station_id": sid, "arrival": T,
        "target_kwh": demand, "delivered_kwh": 0.0,
        "remaining_kwh": demand, "pmax_kw": 6.0,
        "expected_departure": T + pd.Timedelta(minutes=deadline),
    }

def view(cars=()):
    return {"time": T, "capacity_kw": 6.0, "vehicles": tuple(cars)}

cheap_later = lambda t: .4 if t == T else .1
def controller(**kwargs):
    c = RollingHorizonController(
        OUT / f"controller_{len(controllers):03d}",
        kwargs.pop("price_at", cheap_later),
        C_ref=NORM["C_ref"], P_ref=NORM["P_ref"],
        base_hours=kwargs.pop("base_hours", 1), **kwargs)
    controllers.append(c)
    return c

check("uses frozen references",
      abs(NORM["C_ref"] - 2.88) < 1e-12 and
      abs(NORM["P_ref"] - 6.656) < 1e-12)

r = controller().decide(view())
check("empty fleet bypasses LP", r["mode"] == "empty" and not r["command"])

unknown = car()
unknown.update(target_kwh=None, remaining_kwh=None)
r = controller().decide(view([unknown]))
check("unreleased request bypasses LP", r["mode"] == "empty")

r = controller(alpha=1).decide(view([car()]))
check("real LP waits for cheaper slot and executes only first slot",
      r["mode"] == "lp" and not r["command"] and
      r["solve_report"]["status"] == "passed")

r = controller(alpha=1).decide(view([car(deadline=15)]))
check("real LP serves imminent departure",
      r["mode"] == "lp" and
      abs(r["command"].get("a", 0) * .25 - 1.5) <= 1.1e-5)

r = controller(alpha=1).decide(view([car(demand=3, deadline=15)]))
check("service priority preserves unavoidable shortfall",
      r["mode"] == "lp" and
      abs(r["solve_report"]["U_star_kWh"] - 1.5) < 1e-7 and
      r["solve_report"]["U_stage2_kWh"] <= 1.5 + 1.01e-5)

day = str(T.tz_convert("America/Los_Angeles").date())
r = controller(alpha=0).decide(view([car()]), {day: 5.0})
check("real LP respects already-realized daily peak",
      r["mode"] == "lp" and
      r["solve_report"]["peak_by_day_kW"][day] >= 5 - 1e-7)

c = controller(base_hours=24)
case = c.build_case(view([car(deadline=30 * 60)]), {day: 5})
check("visible departure extends effective horizon",
      len(case["slots"]) == 120)
check("new local day does not inherit yesterday peak",
      len(case["past_peak"]) == 2 and
      case["past_peak"][day] == 5 and
      list(case["past_peak"].values())[1] == 0)

v1, v2 = view([car()]), view([car()])
v1["vehicles"][0]["disconnectTime"] = T + pd.Timedelta(hours=1)
v2["vehicles"][0]["disconnectTime"] = T + pd.Timedelta(hours=20)
check("case ignores extraneous future truth",
      c.build_case(v1) == c.build_case(v2))

partial = c.build_case(view([car(deadline=10)]))
expired = c.build_case(view([car(deadline=-15)]))
check("partial estimated slot remains unavailable",
      partial["sessions"][0]["available"] == [])
check("expired estimate extends exactly one slot",
      expired["sessions"][0]["available"] == [0])

with patch("src.rolling_horizon._invoke",
           side_effect=RuntimeError("injected solver failure")):
    r = controller().decide(
        view([car("b", deadline=15), car("a", deadline=30)]))
check("solver failure triggers deterministic feasible EDF",
      r["fallback"] and r["command"] == {"b": 6.0} and
      "injected solver failure" in r["reason"])

r = controller(time_limit_seconds=1e-6).decide(view([car()]))
check("exhausted outer budget triggers feasible fallback",
      r["fallback"] and r["command"] == {"a": 6.0})

peaks = {}
segment = {"start": T, "end": T + pd.Timedelta(minutes=15), "total_kw": 4}
record_realized_peaks(peaks, [segment], segment["end"])
check("peak feedback uses actual executed power", peaks[day] == 4)
try:
    record_realized_peaks({}, [segment], T)
except ValueError:
    check("future feedback rejected", True)
else:
    raise AssertionError("Future feedback accepted")

# Two-step closed-loop execution through the accepted ACN-Sim backend.
sessions = pd.DataFrame([{
    "sessionID": "a", "stationID": "a", "connectionTime": T,
    "cohort": "main_at_arrival", "initial_kWhRequested": 1.5,
    "initial_requestedDeparture": T + pd.Timedelta(minutes=30),
}])
events = pd.DataFrame([
    {"sessionID": "a", "event_type": "arrival", "event_time": T},
    {"sessionID": "a", "event_type": "departure",
     "event_time": T + pd.Timedelta(minutes=30)},
])
executor = ACNSimExecutor(["a"], capacity_kw=6)
e = AggregateEnvironment(
    sessions, events, pd.DataFrame(), start=T,
    capacity_kw=6, executor=executor)
c, peaks, modes = controller(alpha=1), {}, []
for _ in range(2):
    decision = c.decide(e.view(), peaks)
    modes.append(decision["mode"])
    index = len(e.ledger)
    e.step(decision["command"])
    record_realized_peaks(peaks, e.ledger[index:], e.now)
check("RH-LP closed loop uses actual ACN-Sim delivery",
      modes == ["lp", "lp"] and len(e.completed) == 1 and
      abs(e.completed[0]["delivered_kwh"] - 1.5) <= 1.1e-5 and
      abs(executor.evs["a"].energy_delivered -
          e.completed[0]["delivered_kwh"]) < 1e-8)

result = {
    "status": "passed", "check_count": len(passed), "checks": passed,
    "C_ref": NORM["C_ref"], "P_ref": NORM["P_ref"],
    "scope": "RH-LP synthetic validation; development replay pending",
    "timeout_test": "budget guard; not a computation-performance benchmark",
}
Path(sys.argv[1]).write_text(
    json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(result, ensure_ascii=False, indent=2))
'''

files = {
    ROOT / "src/rolling_horizon.py": controller_source.lstrip(),
    ROOT / "tests/test_rolling_horizon.py": test_source.lstrip(),
}
for path, text in files.items():
    compile(text, str(path), "exec")
    if path.exists() and path.read_text(encoding="utf-8") != text:
        raise FileExistsError(f"保留已有不同文件，未覆盖：{path}")
for path, text in files.items():
    path.write_text(text, encoding="utf-8")

output = LOG / "stdout_stderr.txt"
with output.open("w", encoding="utf-8") as stream:
    try:
        result = subprocess.run(
            [sys.executable, str(ROOT / "tests/test_rolling_horizon.py"),
             str(LOG / "verification.json")],
            cwd=ROOT, stdout=stream, stderr=subprocess.STDOUT,
            text=True, timeout=300,
        )
        code = result.returncode
    except subprocess.TimeoutExpired:
        code = "TIMEOUT"

print("解释器：", sys.executable)
print("退出码：", code)
print("证据目录：", LOG)
print(output.read_text(encoding="utf-8"))

解释器： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/.envs/ev-charging/bin/python
退出码： 0
证据目录： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/results/logs/stage3_rolling_synthetic_20261112T081433785555Z
/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/.envs/ev-charging/lib/python3.11/site-packages/acnportal/acnsim/base.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
{
  "status": "passed",
  "check_count": 17,
  "checks": [
    "uses frozen references",
    "empty fleet bypasses LP",
    "unreleased request bypasses LP",
    "real LP waits for cheaper slot and executes only first slot",
    "real LP serves imminent departure",
    "service priority preserves unavoidable shortfall",
    "real LP respects already-realized daily peak",


In [14]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import sys

ROOT = Path("/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
LOG = ROOT / "results/logs" / f"stage3_rolling_replay_{stamp}"
LOG.mkdir(parents=True, exist_ok=False)

source = r'''
from pathlib import Path
from bisect import bisect_right
from collections import Counter
import hashlib
import json
import sys
import numpy as np
import pandas as pd
import yaml

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
from src.rolling_horizon import RollingHorizonController, record_realized_peaks
from src.simulator import AggregateEnvironment
from src.acnsim_adapter import ACNSimExecutor
from src.tariff import attach_tariff

OUT = Path(sys.argv[1])
DATA = ROOT / "data/processed/caltech_12weeks_20261111T143402Z_9b087d"
passed = []

def check(name, condition):
    if not condition:
        raise AssertionError(name)
    passed.append(name)

def config(name):
    return yaml.safe_load(
        (ROOT / "configs" / name).read_text(encoding="utf-8"))

def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

station, splits = config("station.yaml"), config("splits.yaml")
norm_path = ROOT / "configs/normalization.yaml"
norm_before = digest(norm_path)
norm = config("normalization.yaml")
TZ = station["timezone"]
start = pd.Timestamp(splits["splits"]["development"]["start_utc_inclusive"])
dev_end = pd.Timestamp(splits["splits"]["development"]["end_utc_exclusive"])
arrival_end = start + pd.Timedelta(days=7)
capacity = float(station["station_capacity_kW"])
pmax = float(station["evse_max_power_kW"])

check("frozen execution settings",
      station["control_interval_minutes"] == 15 and
      station["charging_efficiency"] == 1 and
      station["network_model"] == "aggregate_continuous" and
      not station["original_branch_constraints_enforced"])
check("formal normalization preserved",
      abs(norm["C_ref"] - 2.88) < 1e-12 and
      abs(norm["P_ref"] - 6.656) < 1e-12)

# This calendar is public/known; it contains no future demand information.
grid = pd.date_range(
    start, dev_end + pd.Timedelta(days=2), freq="15min", inclusive="left")
priced = attach_tariff(pd.DataFrame({"start_utc": grid}))
price_cache = dict(zip(grid.asi8, priced["price_USD_per_kWh"].astype(float)))
def price_at(t):
    t = pd.Timestamp(t)
    if t.value not in price_cache:
        frame = attach_tariff(pd.DataFrame({"start_utc": [t]}))
        price_cache[t.value] = float(frame["price_USD_per_kWh"].iloc[0])
    return price_cache[t.value]

# Independently compare the existing tariff module with the saved tariff table.
saved_tariff = pd.read_csv(
    ROOT / "data/external/tariff.csv",
    usecols=["start_utc", "price_USD_per_kWh"])
saved_tariff["start_utc"] = pd.to_datetime(saved_tariff["start_utc"], utc=True)
sample_grid = pd.date_range(start, arrival_end, freq="15min", inclusive="left")
saved_prices = saved_tariff.set_index("start_utc")[
    "price_USD_per_kWh"].reindex(sample_grid)
check("known tariff agrees with saved table",
      saved_prices.notna().all() and
      np.max(np.abs(saved_prices.to_numpy() -
                    np.array([price_at(t) for t in sample_grid]))) <= 1e-12)

sessions = pd.read_csv(DATA / "sessions_with_split.csv")
sessions["connectionTime"] = pd.to_datetime(sessions["connectionTime"], utc=True)
events = pd.read_csv(DATA / "environment_events.csv")
events["event_time"] = pd.to_datetime(events["event_time"], utc=True)
releases = pd.read_csv(DATA / "input_release_events.csv")
updates = pd.read_csv(DATA / "user_updates.csv")

def valid(value):
    return str(value).strip().lower() in {"true", "1", "1.0"}

summaries = []
for cohort, expected in [
    ("main_at_arrival", 5),
    ("supplement_after_submission", 198),
]:
    selected = sessions[
        (sessions["split"] == "development") &
        (sessions["cohort"] == cohort) &
        (sessions["connectionTime"] >= start) &
        (sessions["connectionTime"] < arrival_end)
    ].copy()
    check(cohort + ": fixed first-week selection", len(selected) == expected)
    ids = set(selected["sessionID"])
    ev = events[events["sessionID"].isin(ids)].copy()
    rel = releases[releases["sessionID"].isin(ids)].copy()
    arrivals = dict(zip(selected["sessionID"], selected["connectionTime"]))
    departures = dict(ev.loc[
        ev["event_type"] == "departure",
        ["sessionID", "event_time"]].itertuples(index=False, name=None))
    end = max(arrival_end, max(departures.values()).ceil("15min"))
    check(cohort + ": tail inside development split", end <= dev_end)

    # Independent target history, from user_updates rather than release events.
    history = {}
    for row in selected.to_dict("records"):
        sid = row["sessionID"]
        if cohort == "main_at_arrival":
            times, targets = [arrivals[sid].value], [float(row["initial_kWhRequested"])]
        else:
            u = updates[updates["sessionID"] == sid].copy()
            u = u[u["valid"].map(valid)]
            u["modifiedAt"] = pd.to_datetime(u["modifiedAt"], utc=True)
            u = u[(u["modifiedAt"] >= arrivals[sid]) &
                  (u["modifiedAt"] < departures[sid])]
            u = u.sort_values(["modifiedAt", "input_order"])
            times = [t.value for t in u["modifiedAt"]]
            targets = u["kWhRequested"].astype(float).tolist()
        assert times
        history[sid] = (times, targets)

    folder = OUT / cohort
    folder.mkdir()
    executor = ACNSimExecutor(
        selected["stationID"].astype(str).tolist(),
        capacity_kw=capacity, voltage=float(station["evse_voltage_V"]),
        max_current=float(station["evse_max_current_A"]))
    e = AggregateEnvironment(
        selected, ev, rel, start=start, capacity_kw=capacity,
        pmax_kw=pmax, executor=executor)
    controller = RollingHorizonController(
        folder / "decisions", price_at,
        C_ref=norm["C_ref"], P_ref=norm["P_ref"],
        alpha=.5, base_hours=24, time_limit_seconds=30, timezone=TZ)
    peaks, decisions = {}, []
    visible_cases = 0

    while e.now < end:
        state = e.view()
        decision = controller.decide(state, dict(peaks))
        report = decision["solve_report"]
        today = str(e.now.tz_convert(TZ).date())
        if decision["mode"] == "lp":
            case = json.loads(
                (Path(decision["step_directory"]) / "case.json").read_text(
                    encoding="utf-8"))
            visible = {
                v["id"]: v for v in state["vehicles"]
                if v["target_kwh"] is not None and
                v["remaining_kwh"] > 1e-9 and
                v["arrival"].ceil("15min") <= e.now
            }
            assert {c["id"] for c in case["sessions"]} == set(visible)
            for c in case["sessions"]:
                assert abs(c["demand"] -
                           visible[c["id"]]["remaining_kwh"]) <= 1e-8
            for day, value in case["past_peak"].items():
                expected_peak = peaks.get(day, 0.0) if day == today else 0.0
                assert abs(value - expected_peak) <= 1e-8
            assert report["peak_by_day_kW"][today] >= peaks.get(today, 0) - 1e-6
            visible_cases += 1

        solver_seconds = (
            sum(s["seconds"] for s in report["stages"])
            if report is not None else None)
        decisions.append({
            "time": e.now, "mode": decision["mode"],
            "total_seconds": decision["total_seconds"],
            "two_stage_solver_seconds": solver_seconds,
            "effective_horizon_hours": decision["effective_horizon_hours"],
            "fallback": decision["fallback"], "reason": decision["reason"],
            "step_directory": decision["step_directory"],
        })
        index = len(e.ledger)
        e.step(decision["command"])
        record_realized_peaks(peaks, e.ledger[index:], e.now, TZ)

        if len(decisions) % 96 == 0:
            progress = {
                "cohort": cohort, "time": str(e.now),
                "steps": len(decisions),
                "fallbacks": sum(d["fallback"] for d in decisions),
            }
            (OUT / "progress.json").write_text(
                json.dumps(progress, indent=2), encoding="utf-8")
            print(json.dumps(progress), flush=True)

    check(cohort + ": LP exercised on real visible requests", visible_cases > 0)
    check(cohort + ": complete selected-session replay",
          len(e.completed) == expected and not e.view()["vehicles"] and
          not executor._connected)

    cumulative = {sid: 0.0 for sid in ids}
    independent_peaks, station_rows, energy_rows = {}, [], []
    energy_residual = power_residual = 0.0
    for segment in e.ledger:
        a, b = segment["start"], segment["end"]
        h = (b - a).value / 3_600_000_000_000
        assert h > 0
        day = str(a.tz_convert(TZ).date())
        assert day == str((b - pd.Timedelta(nanoseconds=1)).tz_convert(TZ).date())
        independent_peaks[day] = max(
            independent_peaks.get(day, 0), segment["total_kw"])
        total_energy = sum(segment["energy_kwh"].values())
        energy_residual = max(
            energy_residual, abs(total_energy - segment["total_kw"] * h))
        power_residual = max(power_residual, segment["total_kw"] - capacity)
        station_rows.append({
            "start": a, "end": b, "total_kw": segment["total_kw"],
            "energy_kwh": total_energy,
        })
        for sid, amount in segment["energy_kwh"].items():
            power = segment["power_kw"][sid]
            power_residual = max(power_residual, power - pmax, -power)
            energy_residual = max(energy_residual, abs(amount - power * h))
            assert a >= arrivals[sid].ceil("15min") and b <= departures[sid]
            times, targets = history[sid]
            j = bisect_right(times, a.value) - 1
            assert j >= 0
            assert a >= pd.Timestamp(times[0], tz="UTC").ceil("15min")
            before = cumulative[sid]
            cumulative[sid] += amount
            assert cumulative[sid] <= max(before, targets[j]) + 1e-5
            energy_rows.append({
                "sessionID": sid, "start": a, "end": b,
                "actual_kw": power, "energy_kwh": amount,
            })

    check(cohort + ": independent input, connection and demand audit", True)
    check(cohort + ": device, station and energy residuals",
          power_residual <= 1e-6 and energy_residual <= 1e-5)
    check(cohort + ": independent actual daily peak feedback",
          peaks.keys() == independent_peaks.keys() and
          all(abs(peaks[d] - independent_peaks[d]) <= 1e-8 for d in peaks))
    check(cohort + ": cumulative ACN-Sim feedback",
          all(abs(cumulative[sid] - executor.evs[sid].energy_delivered) <= 1e-5
              for sid in ids) and
          all(abs(cumulative[s["id"]] - s["delivered_kwh"]) <= 1e-5
              for s in e.completed))

    frame = pd.DataFrame(decisions)
    active = frame[frame["mode"] != "empty"]
    fallbacks = frame[frame["fallback"]]
    check(cohort + ": every decision below control interval",
          frame["total_seconds"].max() < 900)
    frame.to_csv(folder / "decision_summary.csv", index=False)
    pd.DataFrame(station_rows).to_csv(
        folder / "station_trajectory.csv", index=False)
    pd.DataFrame(energy_rows).to_csv(
        folder / "session_trajectory.csv", index=False)
    pd.DataFrame(e.completed).to_csv(
        folder / "completed_sessions.csv", index=False)
    summaries.append({
        "cohort": cohort, "sessions": expected,
        "execution_end_utc": str(end),
        "decision_modes": dict(Counter(frame["mode"])),
        "active_decision_mean_seconds": float(active["total_seconds"].mean()),
        "active_decision_P95_seconds": float(active["total_seconds"].quantile(.95)),
        "active_decision_max_seconds": float(active["total_seconds"].max()),
        "fallback_count": len(fallbacks),
        "fallback_fraction_active": len(fallbacks) / len(active),
        "fallback_examples": fallbacks[["time", "reason"]].head(10).astype(str).to_dict("records"),
        "max_power_residual_kW": power_residual,
        "max_energy_residual_kwh": energy_residual,
        "diagnostic_delivered_kwh": sum(cumulative.values()),
        "requires_review": bool(
            len(fallbacks) or active["total_seconds"].quantile(.95) > 30),
    })

check("normalization file unchanged", digest(norm_path) == norm_before)
tracked = [
    ROOT / "src/rolling_horizon.py", ROOT / "src/optimization_model.py",
    ROOT / "src/simulator.py", ROOT / "src/acnsim_adapter.py",
    ROOT / "src/baselines.py", ROOT / "src/tariff.py", Path(__file__),
    norm_path, ROOT / "configs/station.yaml", ROOT / "configs/splits.yaml",
    DATA / "sessions_with_split.csv", DATA / "environment_events.csv",
    DATA / "input_release_events.csv", DATA / "user_updates.csv",
]
result = {
    "status": "passed", "check_count": len(passed), "checks": passed,
    "replays": summaries,
    "requires_review": any(s["requires_review"] for s in summaries),
    "scope": "development first-week dynamic validation; not formal comparison",
    "source_and_input_sha256": {
        str(p.relative_to(ROOT)): digest(p) for p in tracked},
}
(OUT / "verification.json").write_text(
    json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(
    {k: v for k, v in result.items() if k != "source_and_input_sha256"},
    ensure_ascii=False, indent=2))
'''

path = ROOT / "experiments/run_rolling.py"
text = source.lstrip()
compile(text, str(path), "exec")
if path.exists() and path.read_text(encoding="utf-8") != text:
    raise FileExistsError(f"保留已有不同文件，未覆盖：{path}")
path.write_text(text, encoding="utf-8")

print("开始首周动态回放；进度保存到：", LOG / "progress.json", flush=True)
output = LOG / "stdout_stderr.txt"
with output.open("w", encoding="utf-8") as stream:
    try:
        result = subprocess.run(
            [sys.executable, str(path), str(LOG)],
            cwd=ROOT, stdout=stream, stderr=subprocess.STDOUT,
            text=True, timeout=3600,
        )
        code = result.returncode
    except subprocess.TimeoutExpired:
        code = "TIMEOUT"

print("解释器：", sys.executable)
print("退出码：", code)
print("证据目录：", LOG)
print(output.read_text(encoding="utf-8"))

开始首周动态回放；进度保存到： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/results/logs/stage3_rolling_replay_20261112T082426223430Z/progress.json
解释器： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/.envs/ev-charging/bin/python
退出码： 0
证据目录： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/results/logs/stage3_rolling_replay_20261112T082426223430Z
/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/.envs/ev-charging/lib/python3.11/site-packages/acnportal/acnsim/base.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
{"cohort": "main_at_arrival", "time": "2019-05-07 07:00:00+00:00", "steps": 96, "fallbacks": 0}
{"cohort": "main_at_arrival", "time": "2019-05-08 07:00:00+00:00", "steps": 192, "fallbacks": 0}
{"cohort": "main_at_arrival", "time":

In [15]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import sys

ROOT = Path("/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
LOG = ROOT / "results/logs" / f"stage3_metrics_{stamp}"
LOG.mkdir(parents=True, exist_ok=False)

metrics_source = r'''
"""Metrics from actual execution. Future truth is evaluator-only input."""
from collections import defaultdict
import math
import numpy as np
import pandas as pd

def _time(value):
    value = pd.Timestamp(value)
    if pd.isna(value) or value.tzinfo is None:
        raise ValueError("Timezone-aware timestamp required")
    return value.tz_convert("UTC")

def _number(value, nonnegative=False):
    value = float(value)
    if not math.isfinite(value) or (nonnegative and value < 0):
        raise ValueError("Invalid numeric value")
    return value

def evaluate_run(station_trace, vehicle_trace, sessions, start, end,
                 price_at, capacity_kw, timezone="America/Los_Angeles",
                 service_epsilon=.01, power_tolerance=1e-6):
    """
    station_trace: start,end,total_kw[,energy_kwh]; complete continuous coverage.
    vehicle_trace: sessionID,start,end,actual_kw,energy_kwh;
                   intervals must match station execution segments.
    sessions: sessionID,connectionTime,disconnectTime,evaluation_target_kwh,pmax_kw.
    Optional checkpoint fields:
      first_expected_departure, expected_checkpoint_target_kwh, first_input_time.
    Checkpoint targets are supplied explicitly by the evaluator's input policy.
    """
    start, end = _time(start), _time(end)
    delta = pd.Timedelta(minutes=15)
    if not start < end or start != start.floor(delta) or end != end.floor(delta):
        raise ValueError("Evaluation boundaries must follow the 15-minute grid")
    capacity_kw = _number(capacity_kw, True)
    service_epsilon = _number(service_epsilon, True)

    records = {}
    for row in sessions.to_dict("records"):
        sid = str(row["sessionID"])
        if sid in records:
            raise ValueError("Duplicate session")
        row = dict(row)
        row["connectionTime"] = _time(row["connectionTime"])
        row["disconnectTime"] = _time(row["disconnectTime"])
        if not start <= row["connectionTime"] < row["disconnectTime"] <= end:
            raise ValueError("Censored or invalid evaluation session")
        row["evaluation_target_kwh"] = _number(row["evaluation_target_kwh"], True)
        row["pmax_kw"] = _number(row["pmax_kw"], True)
        records[sid] = row

    grid = tuple(pd.date_range(start, end, freq=delta, inclusive="left"))
    grid_energy = {t.value: 0.0 for t in grid}
    daily = {}
    for t in grid:
        day = str(t.tz_convert(timezone).date())
        if day not in daily:
            daily[day] = {
                "local_date": day, "energy_kwh": 0.0, "cost_USD": 0.0,
                "peak_kW": 0.0, "minimum_kW": float("inf"), "grid_slots": 0,
            }
        daily[day]["grid_slots"] += 1

    station_rows = []
    for row in station_trace.to_dict("records"):
        a, b = _time(row["start"]), _time(row["end"])
        station_rows.append((a, b, _number(row["total_kw"]), row))
    station_rows.sort(key=lambda row: (row[0], row[1]))
    intervals, cursor = {}, start
    energy_residual = station_violation = device_violation = 0.0
    station_violations = device_violations = connection_violations = 0

    for a, b, power, original in station_rows:
        if a != cursor or not a < b <= end:
            raise ValueError("Station trace has a gap, overlap or invalid boundary")
        cursor = b
        h = (b - a).value / 3_600_000_000_000
        energy = power * h
        if "energy_kwh" in original:
            energy_residual = max(
                energy_residual, abs(_number(original["energy_kwh"]) - energy))
        intervals[(a.value, b.value)] = energy
        violation = max(0.0, -power, power - capacity_kw)
        station_violation = max(station_violation, violation)
        station_violations += int(violation > power_tolerance)

        t = a
        while t < b:
            cut = min(b, t.floor(delta) + delta)
            part = power * (cut - t).value / 3_600_000_000_000
            day = str(t.tz_convert(timezone).date())
            rate = _number(price_at(t), True)
            daily[day]["energy_kwh"] += part
            daily[day]["cost_USD"] += part * rate
            daily[day]["peak_kW"] = max(daily[day]["peak_kW"], power)
            daily[day]["minimum_kW"] = min(daily[day]["minimum_kW"], power)
            grid_energy[t.floor(delta).value] += part
            t = cut
    if cursor != end:
        raise ValueError("Station trace does not cover the full evaluation period")

    delivered = defaultdict(float)
    interval_vehicle_energy = defaultdict(float)
    vehicle_rows = defaultdict(list)
    seen = set()
    for row in vehicle_trace.to_dict("records"):
        sid = str(row["sessionID"])
        a, b = _time(row["start"]), _time(row["end"])
        key = (a.value, b.value)
        if sid not in records or key not in intervals or (sid, key) in seen:
            raise ValueError("Invalid or duplicate vehicle execution segment")
        seen.add((sid, key))
        power, energy = _number(row["actual_kw"]), _number(row["energy_kwh"])
        h = (b - a).value / 3_600_000_000_000
        energy_residual = max(energy_residual, abs(energy - power * h))
        delivered[sid] += energy
        interval_vehicle_energy[key] += energy
        vehicle_rows[sid].append((a, b, power))
        r = records[sid]
        violation = max(0.0, -power, power - r["pmax_kw"])
        device_violation = max(device_violation, violation)
        device_violations += int(violation > power_tolerance)
        if power > power_tolerance and not (
            a >= r["connectionTime"].ceil(delta) and b <= r["disconnectTime"]
        ):
            connection_violations += 1
    for key, energy in intervals.items():
        energy_residual = max(
            energy_residual, abs(energy - interval_vehicle_energy[key]))

    service_rows = []
    expected_evaluable = expected_uncompleted = 0
    for sid, row in records.items():
        target, actual = row["evaluation_target_kwh"], delivered[sid]
        result = {
            "sessionID": sid, "evaluation_target_kwh": target,
            "delivered_kwh": actual, "unmet_kwh": max(target - actual, 0.0),
            "excess_kwh": max(actual - target, 0.0),
            "satisfaction": min(max(actual, 0.0), target) / target if target > 0 else None,
            "uncompleted_at_actual_departure": actual < target - service_epsilon,
            "expected_checkpoint_evaluable": False,
            "uncompleted_at_expected_departure": None,
        }
        deadline = row.get("first_expected_departure")
        checkpoint_target = row.get("expected_checkpoint_target_kwh")
        if deadline is not None and pd.notna(deadline) and (
            checkpoint_target is not None and pd.notna(checkpoint_target)
        ):
            deadline = _time(deadline)
            first_input = row.get("first_input_time", row["connectionTime"])
            if pd.notna(first_input) and deadline >= _time(first_input):
                checkpoint_target = _number(checkpoint_target, True)
                before = sum(
                    power * max(0, (min(b, deadline) - a).value)
                    / 3_600_000_000_000
                    for a, b, power in vehicle_rows[sid] if a < deadline)
                unfinished = before < checkpoint_target - service_epsilon
                expected_evaluable += 1
                expected_uncompleted += int(unfinished)
                result.update(
                    expected_checkpoint_evaluable=True,
                    uncompleted_at_expected_departure=unfinished)
        service_rows.append(result)

    services = pd.DataFrame(service_rows)
    targets = [r["evaluation_target_kwh"] for r in service_rows]
    total_target = sum(targets)
    fulfilled = sum(
        min(max(r["delivered_kwh"], 0.0), r["evaluation_target_kwh"])
        for r in service_rows)
    fractions = [r["satisfaction"] for r in service_rows if r["satisfaction"] is not None]
    unfinished = sum(r["uncompleted_at_actual_departure"] for r in service_rows)
    daily_rows = list(daily.values())
    grid_power = np.array([grid_energy[t.value] / .25 for t in grid])
    for row in daily_rows:
        row["peak_valley_kW"] = row["peak_kW"] - row["minimum_kW"]
        values = [grid_power[j] for j, t in enumerate(grid)
                  if str(t.tz_convert(timezone).date()) == row["local_date"]]
        row["grid_average_peak_kW"] = float(max(values))

    summary = {
        "cost_USD": sum(r["cost_USD"] for r in daily_rows),
        "delivered_kwh": sum(delivered.values()),
        "unmet_kwh": sum(r["unmet_kwh"] for r in service_rows),
        "excess_kwh": sum(r["excess_kwh"] for r in service_rows),
        "energy_satisfaction": fulfilled / total_target if total_target > 0 else None,
        "sessions": len(records), "uncompleted_sessions": int(unfinished),
        "actual_departure_uncompleted_rate": unfinished / len(records) if records else None,
        "expected_checkpoint_evaluable_sessions": expected_evaluable,
        "expected_checkpoint_excluded_sessions": len(records) - expected_evaluable,
        "expected_checkpoint_uncompleted_sessions": expected_uncompleted,
        "expected_departure_uncompleted_rate": (
            expected_uncompleted / expected_evaluable if expected_evaluable else None),
        "max_individual_unmet_kwh": max([0.0] + [r["unmet_kwh"] for r in service_rows]),
        "session_satisfaction_P10": float(np.quantile(fractions, .1)) if fractions else None,
        "max_peak_kW": max(r["peak_kW"] for r in daily_rows),
        "max_grid_average_peak_kW": float(max(grid_power)),
        "max_grid_load_change_kW": float(max(np.abs(np.diff(grid_power)))) if len(grid_power) > 1 else 0.0,
        "station_violation_segments": station_violations,
        "device_violation_segments": device_violations,
        "connection_violation_segments": connection_violations,
        "max_power_violation_kW": max(station_violation, device_violation),
        "max_energy_residual_kwh": energy_residual,
    }
    return summary, pd.DataFrame(daily_rows), services

def summarize_decisions(frame):
    if frame.empty:
        return {"decision_count": 0, "fallback_count": 0,
                "active_P95_seconds": None, "active_max_seconds": None}
    active = frame[frame["mode"] != "empty"]
    timing = [_number(v, True) for v in active["total_seconds"]]
    fallback = sum(str(v).lower() in {"true", "1", "1.0"} for v in frame["fallback"])
    return {
        "decision_count": len(frame), "active_decision_count": len(active),
        "fallback_count": fallback,
        "fallback_fraction_active": fallback / len(active) if len(active) else None,
        "active_mean_seconds": float(np.mean(timing)) if timing else None,
        "active_P95_seconds": float(np.quantile(timing, .95)) if timing else None,
        "active_max_seconds": max(timing) if timing else None,
        "mode_counts": {str(k): int(v) for k, v in frame["mode"].value_counts().items()},
    }
'''

test_source = r'''
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
from src.metrics import evaluate_run, summarize_decisions

T = pd.Timestamp("2019-05-06T07:00:00Z")
passed = []
def check(name, condition):
    if not condition:
        raise AssertionError(name)
    passed.append(name)

def fixture(powers=(6, 0), target=1.5, start=T):
    end = start + pd.Timedelta(minutes=15 * len(powers))
    station, vehicle = [], []
    for j, power in enumerate(powers):
        a = start + pd.Timedelta(minutes=15 * j)
        b = a + pd.Timedelta(minutes=15)
        station.append({"start": a, "end": b, "total_kw": power,
                        "energy_kwh": power * .25})
        if power:
            vehicle.append({"sessionID": "a", "start": a, "end": b,
                            "actual_kw": power, "energy_kwh": power * .25})
    sessions = [{
        "sessionID": "a", "connectionTime": start, "disconnectTime": end,
        "evaluation_target_kwh": target, "pmax_kw": 10,
    }]
    return pd.DataFrame(station), pd.DataFrame(vehicle), pd.DataFrame(sessions), start, end

def evaluate(args, capacity=10):
    return evaluate_run(*args, price_at=lambda t: .2, capacity_kw=capacity)

s, daily, vehicles = evaluate(fixture())
check("hand-calculated cost and energy",
      abs(s["cost_USD"] - .3) < 1e-10 and abs(s["delivered_kwh"] - 1.5) < 1e-10)
check("zero-load segments retained in peak-valley",
      daily.iloc[0]["peak_kW"] == 6 and daily.iloc[0]["peak_valley_kW"] == 6)
check("fulfilled request service metrics",
      s["energy_satisfaction"] == 1 and s["uncompleted_sessions"] == 0)
check("unspecified expected checkpoint remains NA",
      s["expected_departure_uncompleted_rate"] is None and
      s["expected_checkpoint_excluded_sessions"] == 1)

args = list(fixture(powers=(0, 6)))
args[2]["first_expected_departure"] = T + pd.Timedelta(minutes=15)
args[2]["expected_checkpoint_target_kwh"] = 1.5
s, _, _ = evaluate(args)
check("expected and actual departure indicators remain separate",
      s["actual_departure_uncompleted_rate"] == 0 and
      s["expected_departure_uncompleted_rate"] == 1)

args = list(fixture(powers=(8,), target=1))
extra = args[2].iloc[0].to_dict()
extra["sessionID"] = "b"
args[2] = pd.concat([args[2], pd.DataFrame([extra])], ignore_index=True)
s, _, _ = evaluate(args)
check("excess cannot hide another vehicle shortfall",
      s["energy_satisfaction"] == .5 and s["unmet_kwh"] == 1 and
      s["excess_kwh"] == 1)

args = list(fixture(powers=(0,), target=0))
s, _, _ = evaluate(args)
check("zero total demand satisfaction is NA", s["energy_satisfaction"] is None)

args = list(fixture(powers=(0,)))
args[2] = args[2].iloc[:0]
s, _, _ = evaluate(args)
check("empty fleet rates are NA",
      s["actual_departure_uncompleted_rate"] is None and s["sessions"] == 0)

a, cut, end = T, T + pd.Timedelta(minutes=5), T + pd.Timedelta(minutes=15)
station = pd.DataFrame([
    {"start": a, "end": cut, "total_kw": 6},
    {"start": cut, "end": end, "total_kw": 0},
])
trace = pd.DataFrame([{
    "sessionID": "a", "start": a, "end": cut, "actual_kw": 6, "energy_kwh": .5}])
sessions = fixture(powers=(0,), target=.5)[2]
s, _, _ = evaluate((station, trace, sessions, a, end))
check("instantaneous and grid-average peaks distinguished",
      s["max_peak_kW"] == 6 and abs(s["max_grid_average_peak_kW"] - 2) < 1e-10)

late = pd.Timestamp("2019-05-06T23:45:00", tz="America/Los_Angeles").tz_convert("UTC")
s, daily, _ = evaluate(fixture(start=late))
check("local midnight assigns energy to correct day",
      len(daily) == 2 and abs(daily.iloc[0]["energy_kwh"] - 1.5) < 1e-10 and
      daily.iloc[1]["energy_kwh"] == 0)

for date, duration in [("2019-03-10", 23), ("2019-11-03", 25)]:
    local = pd.Timestamp(date, tz="America/Los_Angeles")
    a = local.tz_convert("UTC")
    b = (local + pd.DateOffset(days=1)).tz_convert("UTC")
    station = pd.DataFrame([{"start": a, "end": b, "total_kw": 6}])
    trace = pd.DataFrame([{
        "sessionID": "a", "start": a, "end": b,
        "actual_kw": 6, "energy_kwh": 6 * duration}])
    sessions = pd.DataFrame([{
        "sessionID": "a", "connectionTime": a, "disconnectTime": b,
        "evaluation_target_kwh": 500, "pmax_kw": 10}])
    s, daily, _ = evaluate((station, trace, sessions, a, b))
    check(f"DST {duration}-hour daily integration",
          abs(s["delivered_kwh"] - 6 * duration) < 1e-8 and
          len(daily) == 1 and daily.iloc[0]["grid_slots"] == duration * 4)

args = list(fixture(powers=(7,)))
args[2]["pmax_kw"] = 6
s, _, _ = evaluate(args, capacity=6)
check("constraint violations are reported",
      s["station_violation_segments"] == 1 and
      s["device_violation_segments"] == 1 and s["max_power_violation_kW"] == 1)

args = list(fixture())
args[0] = args[0].iloc[:1]
try:
    evaluate(args)
except ValueError:
    check("missing zero-load coverage rejected", True)
else:
    raise AssertionError("Incomplete coverage accepted")

timing = summarize_decisions(pd.DataFrame([
    {"mode": "empty", "total_seconds": .01, "fallback": "False"},
    {"mode": "lp", "total_seconds": 2, "fallback": "False"},
    {"mode": "fallback", "total_seconds": 3, "fallback": "True"},
]))
check("timing excludes empty steps and parses CSV booleans",
      timing["fallback_count"] == 1 and timing["active_mean_seconds"] == 2.5)

result = {
    "status": "passed", "check_count": len(passed), "checks": passed,
    "scope": "metrics hand-calculation checks; four-method integration pending",
}
Path(sys.argv[1]).write_text(
    json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(result, ensure_ascii=False, indent=2))
'''

files = {
    ROOT / "src/metrics.py": metrics_source.lstrip(),
    ROOT / "tests/test_metrics.py": test_source.lstrip(),
}
for path, text in files.items():
    compile(text, str(path), "exec")
    if path.exists() and path.read_text(encoding="utf-8") != text:
        raise FileExistsError(f"保留已有不同文件，未覆盖：{path}")
for path, text in files.items():
    path.write_text(text, encoding="utf-8")

output = LOG / "stdout_stderr.txt"
with output.open("w", encoding="utf-8") as stream:
    try:
        result = subprocess.run(
            [sys.executable, str(ROOT / "tests/test_metrics.py"),
             str(LOG / "verification.json")],
            cwd=ROOT, stdout=stream, stderr=subprocess.STDOUT,
            text=True, timeout=180,
        )
        code = result.returncode
    except subprocess.TimeoutExpired:
        code = "TIMEOUT"

print("解释器：", sys.executable)
print("退出码：", code)
print("证据目录：", LOG)
print(output.read_text(encoding="utf-8"))

解释器： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/.envs/ev-charging/bin/python
退出码： 0
证据目录： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/results/logs/stage3_metrics_20261112T085600815670Z
{
  "status": "passed",
  "check_count": 15,
  "checks": [
    "hand-calculated cost and energy",
    "zero-load segments retained in peak-valley",
    "fulfilled request service metrics",
    "unspecified expected checkpoint remains NA",
    "expected and actual departure indicators remain separate",
    "excess cannot hide another vehicle shortfall",
    "zero total demand satisfaction is NA",
    "empty fleet rates are NA",
    "instantaneous and grid-average peaks distinguished",
    "local midnight assigns energy to correct day",
    "DST 23-hour daily integration",
    "DST 25-hour daily integration",
    "constraint violations are reported",
    "missing zero-load coverage rejected",
    "timing excludes empty steps and parses CSV booleans"
  ],
  "scope": "metrics h

In [16]:
from pathlib import Path
import hashlib
import json

ROOT = Path("/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization")
EVIDENCE = ROOT / (
    "results/logs/stage3_rolling_replay_20261112T082426223430Z"
)

print("STAGE3_INTEGRATION_PRECHECK")

print("\n===== 已验收模型规格 =====")
model = ROOT / "report/model.md"
if not model.is_file():
    raise FileNotFoundError(model)
print(model.read_text(encoding="utf-8"))

print("\n===== RH-LP 轨迹复用检查 =====")
record = json.loads(
    (EVIDENCE / "verification.json").read_text(encoding="utf-8")
)
assert record["status"] == "passed"
assert record["requires_review"] is False

mismatches = []
for relative, expected in record["source_and_input_sha256"].items():
    path = ROOT / relative
    actual = (
        hashlib.sha256(path.read_bytes()).hexdigest()
        if path.is_file() else "MISSING"
    )
    if actual != expected:
        mismatches.append(relative)

print("代码及输入哈希不匹配项：", mismatches)

for cohort in ["main_at_arrival", "supplement_after_submission"]:
    for filename in [
        "station_trajectory.csv",
        "session_trajectory.csv",
        "completed_sessions.csv",
        "decision_summary.csv",
    ]:
        path = EVIDENCE / cohort / filename
        assert path.is_file(), f"缺失：{path}"

print("复用检查：", "通过" if not mismatches else "需重新核对，暂不复用")
print("STAGE3_INTEGRATION_PRECHECK_END")

STAGE3_INTEGRATION_PRECHECK

===== 已验收模型规格 =====
# 数学模型规格

状态：阶段二规格、合成验证及开发集严格主队列静态核对已完成；正式归一化数值按第 9 节在阶段三冻结。

## 1. 范围与固定参数

研究对象为 Caltech 单站聚合连续功率充电模型。
车辆连接固定，不建模桩分配、V2G、建筑负荷、电池 SOC 或原网络相位约束。
50 kW 是实验容量，不是真实供电容量声明。

| 符号 | 定义 | 单位及来源 |
|---|---|---|
| Δt | 控制时段长度 | 0.25 h，阶段一冻结 |
| P_i^max | 车辆所连接设备的功率上限 | 6.656 kW，阶段一设备约定 |
| P_t^station | 站点聚合容量 | 基准 50 kW，实验情景 |
| η | 能量转换效率 | 1，电网侧建模约定 |
| c_t | 时段电价 | USD/kWh，冻结实验 TOU |
| α | 费用权重 | 基准 0.5，计划值 |
| C_ref | 费用归一化参考 | USD，正值，待确定并冻结 |
| P_ref | 日峰值归一化参考 | kW，正值，待确定并冻结 |
| ε_U | 第二层允许的总缺口增量 | 初始 1e-5 kWh，待数值核查 |
| ε_service | 服务完成评价容差 | 0.01 kWh，区别于求解容差 |

UTC 用于时间运算；America/Los_Angeles 用于电价和自然日归属。
不强制每个当地自然日具有 96 个时段。

## 2. 信息集与服务目标

### 严格主队列
到达时有效的总请求和预计离站固定用于在线调度；
后续修改仅作审计。评价目标为到达时有效总请求。

### 晚提交补充队列
有效输入在 modifiedAt 生效后才能使用；在控制时刻 k，
仅使用不晚于 k 的最新有效输入。
首次输入生效前不安排供电。
请求是会话累计总目标，不是本次新增请求电量。

补充队列事后评价目标定义为：
连接期间已生效的最后一条有效总请求，按阶段一去重与排序规则确定。
该目标供离线参考与事后评价使用，不能提前传给在线调度器。
同一会话的四方法使用相同评价目标。

设 Q_i(k) 为当前可见总请求，D_i(k) 为本次仿真实际累计交付：
R_i(k) 

In [17]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import sys

ROOT = Path("/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
LOG = ROOT / "results/logs" / f"stage3_integration_{stamp}"
LOG.mkdir(parents=True, exist_ok=False)

source = r'''
from pathlib import Path
from bisect import bisect_right
from time import perf_counter
import hashlib
import json
import shutil
import sys
import numpy as np
import pandas as pd
import yaml

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
from src.baselines import uncontrolled, tou_greedy
from src.simulator import AggregateEnvironment
from src.acnsim_adapter import ACNSimExecutor
from src.metrics import evaluate_run, summarize_decisions
from src.rolling_horizon import _invoke, _first_command
from src.tariff import attach_tariff

OUT = Path(sys.argv[1])
DATA = ROOT / "data/processed/caltech_12weeks_20261111T143402Z_9b087d"
RH = ROOT / "results/logs/stage3_rolling_replay_20261112T082426223430Z"
passed, comparisons, artifact_hashes = [], [], {}
started = perf_counter()

def check(name, condition):
    if not condition:
        raise AssertionError(name)
    passed.append(name)

def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def config(name):
    return yaml.safe_load(
        (ROOT / "configs" / name).read_text(encoding="utf-8"))

station, splits, norm = (
    config("station.yaml"), config("splits.yaml"), config("normalization.yaml"))
norm_hash = digest(ROOT / "configs/normalization.yaml")
TZ = station["timezone"]
capacity = float(station["station_capacity_kW"])
pmax = float(station["evse_max_power_kW"])
start = pd.Timestamp(splits["splits"]["development"]["start_utc_inclusive"])
dev_end = pd.Timestamp(splits["splits"]["development"]["end_utc_exclusive"])
arrival_end = start + pd.Timedelta(days=7)

old = json.loads((RH / "verification.json").read_text(encoding="utf-8"))
check("accepted RH evidence", old["status"] == "passed" and not old["requires_review"])
for relative, expected in old["source_and_input_sha256"].items():
    check("RH source/input unchanged: " + relative,
          digest(ROOT / relative) == expected)
check("frozen scope and references",
      station["network_model"] == "aggregate_continuous" and
      station["control_interval_minutes"] == 15 and
      station["charging_efficiency"] == 1 and
      not station["original_branch_constraints_enforced"] and
      abs(norm["C_ref"] - 2.88) < 1e-12 and
      abs(norm["P_ref"] - 6.656) < 1e-12)

grid = pd.date_range(start, dev_end + pd.Timedelta(days=2),
                     freq="15min", inclusive="left")
priced = attach_tariff(pd.DataFrame({"start_utc": grid}))
price_cache = dict(zip(grid.asi8, priced["price_USD_per_kWh"]))
def price_at(t):
    t = pd.Timestamp(t)
    if t.value not in price_cache:
        price_cache[t.value] = float(attach_tariff(
            pd.DataFrame({"start_utc": [t]}))["price_USD_per_kWh"].iloc[0])
    return float(price_cache[t.value])

sessions = pd.read_csv(DATA / "sessions_with_split.csv")
sessions["connectionTime"] = pd.to_datetime(sessions["connectionTime"], utc=True)
events = pd.read_csv(DATA / "environment_events.csv")
events["event_time"] = pd.to_datetime(events["event_time"], utc=True)
releases = pd.read_csv(DATA / "input_release_events.csv")
updates = pd.read_csv(DATA / "user_updates.csv")

def valid(value):
    return str(value).strip().lower() in {"true", "1", "1.0"}

def change(previous, current):
    if previous is None or current is None:
        return None
    old_slots, old_power = previous
    new_slots, new_power = current
    common = old_slots & new_slots
    if not common:
        return None
    keys = {key for key in old_power.keys() | new_power.keys()
            if key[1] in common}
    return sum(abs(new_power.get(key, 0) - old_power.get(key, 0)) for key in keys)

def tou_bundle(result):
    labels = [t.isoformat() for t in result["slots"]]
    powers = {
        (sid, labels[j]): float(power)
        for sid, values in result["plan_kw"].items()
        for j, power in enumerate(values) if power > 0
    }
    return set(labels), powers

def csv_bundle(path):
    frame = pd.read_csv(path, dtype={"sessionID": str, "slot": str})
    powers = {
        (row.sessionID, row.slot): float(row.power_kW)
        for row in frame.itertuples(index=False) if row.power_kW != 0
    }
    return set(frame["slot"]), powers

def execution_frames(environment):
    station_rows, vehicle_rows = [], []
    for row in environment.ledger:
        station_rows.append({
            "start": row["start"], "end": row["end"],
            "total_kw": row["total_kw"],
            "energy_kwh": sum(row["energy_kwh"].values()),
        })
        for sid, energy in row["energy_kwh"].items():
            vehicle_rows.append({
                "sessionID": sid, "start": row["start"], "end": row["end"],
                "actual_kw": row["power_kw"][sid], "energy_kwh": energy,
            })
    return pd.DataFrame(station_rows), pd.DataFrame(
        vehicle_rows,
        columns=["sessionID", "start", "end", "actual_kw", "energy_kwh"])

for cohort, expected_count in [
    ("main_at_arrival", 5), ("supplement_after_submission", 198),
]:
    selected = sessions[
        (sessions["split"] == "development") &
        (sessions["cohort"] == cohort) &
        (sessions["connectionTime"] >= start) &
        (sessions["connectionTime"] < arrival_end)
    ].copy()
    check(cohort + ": fixed shared selection", len(selected) == expected_count)
    ids = set(selected["sessionID"])
    ev = events[events["sessionID"].isin(ids)].copy()
    rel = releases[releases["sessionID"].isin(ids)].copy()
    departures = dict(ev.loc[
        ev["event_type"] == "departure",
        ["sessionID", "event_time"]].itertuples(index=False, name=None))
    arrivals = dict(zip(selected["sessionID"], selected["connectionTime"]))
    end = max(arrival_end, max(departures.values()).ceil("15min"))
    check(cohort + ": shared tail inside development", end <= dev_end)
    prior_summary = next(r for r in old["replays"] if r["cohort"] == cohort)
    check(cohort + ": RH selection and tail match",
          prior_summary["sessions"] == expected_count and
          pd.Timestamp(prior_summary["execution_end_utc"]) == end)

    # One evaluator-only table is shared by all four methods.
    evaluation_rows, histories = [], {}
    for row in selected.to_dict("records"):
        sid = row["sessionID"]
        if cohort == "main_at_arrival":
            times = [arrivals[sid].value]
            targets = [float(row["initial_kWhRequested"])]
            deadline = pd.Timestamp(row["initial_requestedDeparture"])
        else:
            u = updates[updates["sessionID"] == sid].copy()
            u = u[u["valid"].map(valid)]
            u["modifiedAt"] = pd.to_datetime(u["modifiedAt"], utc=True)
            u = u[(u["modifiedAt"] >= arrivals[sid]) &
                  (u["modifiedAt"] < departures[sid])]
            u = u.sort_values(["modifiedAt", "input_order"])
            assert len(u)
            times = [t.value for t in u["modifiedAt"]]
            targets = u["kWhRequested"].astype(float).tolist()
            deadline = pd.Timestamp(u.iloc[0]["requestedDeparture"])
        histories[sid] = (times, targets)
        checkpoint = bisect_right(times, deadline.value) - 1
        evaluation_rows.append({
            "sessionID": sid, "connectionTime": arrivals[sid],
            "disconnectTime": departures[sid],
            "evaluation_target_kwh": targets[-1], "pmax_kw": pmax,
            "first_input_time": pd.Timestamp(times[0], tz="UTC"),
            "first_expected_departure": deadline,
            "expected_checkpoint_target_kwh": (
                targets[checkpoint] if checkpoint >= 0 else None),
        })
    evaluation = pd.DataFrame(evaluation_rows)
    cohort_dir = OUT / cohort
    cohort_dir.mkdir()
    evaluation.to_csv(cohort_dir / "shared_evaluation_sessions.csv", index=False)
    targets_by_id = evaluation.set_index("sessionID")["evaluation_target_kwh"]

    for method in ["Uncontrolled", "TOU_Greedy", "Static_LP", "RH_LP"]:
        print(f"Integrating {cohort}/{method}", flush=True)
        folder = cohort_dir / method
        folder.mkdir()
        offline_seconds, static_report = None, None

        if method == "RH_LP":
            # Reuse accepted execution; audit it again with the common evaluator.
            for filename in [
                "station_trajectory.csv", "session_trajectory.csv",
                "completed_sessions.csv", "decision_summary.csv",
            ]:
                shutil.copy2(RH / cohort / filename, folder / filename)
            station_frame = pd.read_csv(folder / "station_trajectory.csv")
            vehicle_frame = pd.read_csv(
                folder / "session_trajectory.csv", dtype={"sessionID": str})
            decisions = pd.read_csv(folder / "decision_summary.csv")
            previous, changes = None, []
            for row in decisions.to_dict("records"):
                current = (
                    csv_bundle(Path(row["step_directory"]) / "lp/plan.csv")
                    if row["mode"] == "lp" else None)
                changes.append(change(previous, current))
                previous = current
            decisions["plan_change_kW_sum"] = changes
        else:
            runtime_sessions = selected.copy()
            plan_by_time = {}
            if method == "Static_LP":
                # Explicit full-information oracle initialization.
                runtime_sessions["cohort"] = "main_at_arrival"
                runtime_sessions["initial_kWhRequested"] = (
                    runtime_sessions["sessionID"].map(targets_by_id))
                runtime_sessions["initial_requestedDeparture"] = (
                    runtime_sessions["sessionID"].map(departures))
                horizon = tuple(pd.date_range(
                    start, end, freq="15min", inclusive="left"))
                case = {
                    "dt": .25, "alpha": .5,
                    "C_ref": norm["C_ref"], "P_ref": norm["P_ref"],
                    "epsilon_U": 1e-5, "time_limit_seconds": 120,
                    "past_peak": {},
                    "slots": [
                        {"label": t.isoformat(),
                         "day": str(t.tz_convert(TZ).date()),
                         "price": price_at(t), "capacity": capacity}
                        for t in horizon
                    ],
                    "sessions": [
                        {"id": sid, "demand": float(targets_by_id[sid]),
                         "pmax": pmax, "available": list(range(
                             int(pd.Index(horizon).searchsorted(
                                 arrivals[sid].ceil("15min"))),
                             int(pd.Index(horizon).searchsorted(
                                 departures[sid].floor("15min")))))}
                        for sid in sorted(ids)
                    ],
                }
                static_dir = folder / "static_solve"
                static_dir.mkdir()
                tick = perf_counter()
                static_report = _invoke(case, static_dir, 120)
                _first_command(case, static_dir / "lp/plan.csv")
                plan = pd.read_csv(
                    static_dir / "lp/plan.csv",
                    dtype={"sessionID": str, "slot": str})
                for row in plan.itertuples(index=False):
                    if row.power_kW > 0:
                        plan_by_time.setdefault(row.slot, {})[
                            row.sessionID] = float(row.power_kW)
                offline_seconds = perf_counter() - tick

            executor = ACNSimExecutor(
                selected["stationID"].astype(str).tolist(),
                capacity_kw=capacity, voltage=float(station["evse_voltage_V"]),
                max_current=float(station["evse_max_current_A"]))
            environment = AggregateEnvironment(
                runtime_sessions, ev,
                pd.DataFrame() if method == "Static_LP" else rel,
                start=start, capacity_kw=capacity, pmax_kw=pmax,
                executor=executor)
            rows, previous = [], None
            with (folder / "requests_and_plans.jsonl").open(
                "w", encoding="utf-8"
            ) as journal:
                while environment.now < end:
                    tick = perf_counter()
                    state = environment.view()
                    current, diagnostic = None, {}
                    if method == "Uncontrolled":
                        diagnostic = uncontrolled(state, details=True)
                        command = diagnostic["command"]
                    elif method == "TOU_Greedy":
                        result = tou_greedy(state, price_at, details=True)
                        command = result["command"]
                        current = tou_bundle(result)
                        diagnostic = {
                            "forecast_unmet_kwh": result["forecast_unmet_kwh"],
                            "effective_horizon_hours": result["effective_horizon_hours"],
                            "sparse_plan": [
                                [sid, label, power]
                                for (sid, label), power in current[1].items()],
                        }
                    else:
                        raw = plan_by_time.get(environment.now.isoformat(), {})
                        active_ids = {v["id"] for v in state["vehicles"]}
                        assert set(raw).issubset(active_ids)
                        command, left = {}, capacity
                        for v in sorted(state["vehicles"], key=lambda v: v["id"]):
                            raw_power = raw.get(v["id"], 0.0)
                            power = min(raw_power, v["pmax_kw"],
                                        v["remaining_kwh"] / .25, left)
                            assert abs(power - raw_power) <= 1e-6
                            if power > 0:
                                command[v["id"]] = power
                                left = max(left - power, 0)
                    active = any(
                        v["remaining_kwh"] is not None and v["remaining_kwh"] > 1e-9
                        for v in state["vehicles"])
                    journal.write(json.dumps({
                        "time": str(environment.now), "command": command,
                        "diagnostic": diagnostic,
                    }, default=str) + "\n")
                    rows.append({
                        "time": environment.now,
                        "mode": ("static_replay" if method == "Static_LP"
                                 else "rule") if active else "empty",
                        "total_seconds": perf_counter() - tick,
                        "fallback": False,
                        "plan_change_kW_sum": (
                            0.0 if method == "Static_LP" else change(previous, current)),
                    })
                    previous = current
                    environment.step(command)
            check(f"{cohort}/{method}: complete shared-environment replay",
                  len(environment.completed) == expected_count and
                  not environment.view()["vehicles"] and not executor._connected)
            station_frame, vehicle_frame = execution_frames(environment)
            decisions = pd.DataFrame(rows)
            station_frame.to_csv(folder / "station_trajectory.csv", index=False)
            vehicle_frame.to_csv(folder / "session_trajectory.csv", index=False)
            pd.DataFrame(environment.completed).to_csv(
                folder / "completed_sessions.csv", index=False)

        # All online traces must obey the request history at the time of delivery.
        if method != "Static_LP":
            cumulative = {sid: 0.0 for sid in ids}
            trace = vehicle_frame.copy()
            trace["start"] = pd.to_datetime(trace["start"], utc=True)
            trace = trace.sort_values(["start", "sessionID"])
            for row in trace.to_dict("records"):
                sid, amount = row["sessionID"], float(row["energy_kwh"])
                if amount <= 0:
                    continue
                times, targets = histories[sid]
                j = bisect_right(times, row["start"].value) - 1
                assert j >= 0
                assert row["start"] >= pd.Timestamp(times[0], tz="UTC").ceil("15min")
                before = cumulative[sid]
                cumulative[sid] += amount
                assert cumulative[sid] <= max(before, targets[j]) + 1e-5
            check(f"{cohort}/{method}: online information audit", True)

        summary, daily, service = evaluate_run(
            station_frame, vehicle_frame, evaluation, start, end,
            price_at, capacity_kw=capacity, timezone=TZ)
        check(f"{cohort}/{method}: unified physical and energy checks",
              summary["max_power_violation_kW"] <= 1e-6 and
              summary["max_energy_residual_kwh"] <= 1e-5 and
              summary["connection_violation_segments"] == 0)
        check(f"{cohort}/{method}: shared target and session accounting",
              summary["sessions"] == expected_count and
              set(service["sessionID"]) == ids)
        if method == "RH_LP":
            check(cohort + ": reused RH energy reconciles with accepted report",
                  abs(summary["delivered_kwh"] -
                      prior_summary["diagnostic_delivered_kwh"]) <= 1e-5)
        if method == "Static_LP":
            check(cohort + ": Static planned and actual shortfall agree",
                  abs(summary["unmet_kwh"] - static_report["U_stage2_kWh"]) <= 1e-5)

        timing = summarize_decisions(decisions)
        check(f"{cohort}/{method}: no unexpected fallback",
              timing["fallback_count"] == 0)
        changes = pd.to_numeric(decisions["plan_change_kW_sum"], errors="coerce").dropna()
        summary.update(
            cohort=cohort, method=method,
            information=("offline_full_information_oracle"
                         if method == "Static_LP" else "online_visible_inputs"),
            rh_reused=(method == "RH_LP"),
            offline_solve_wall_seconds=offline_seconds,
            plan_change_mean_kW_sum=float(changes.mean()) if len(changes) else None,
            plan_change_pairs=int(len(changes)),
            constraint_violation_duration_hours=0.0,
            **timing)
        comparisons.append(summary)
        daily.to_csv(folder / "daily_metrics.csv", index=False)
        service.to_csv(folder / "session_service.csv", index=False)
        decisions.to_csv(folder / "decision_summary.csv", index=False)
        (folder / "summary.json").write_text(
            json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
        for filename in [
            "station_trajectory.csv", "session_trajectory.csv",
            "session_service.csv", "daily_metrics.csv", "decision_summary.csv",
        ]:
            path = folder / filename
            artifact_hashes[str(path.relative_to(OUT))] = digest(path)

check("normalization unchanged",
      digest(ROOT / "configs/normalization.yaml") == norm_hash)
check("all eight method/cohort combinations completed", len(comparisons) == 8)
table = pd.DataFrame(comparisons)
table.to_csv(OUT / "comparison.csv", index=False)

policy = """# Stage 3 integrated development validation

Evaluation uses the same selected first-week arrivals and full departure tail
for all four methods, with separate cohorts and initial delivered energy zero.

Main target: frozen arrival request.
Supplement target: last valid request released during connection, before departure.
Same-time input conflicts: ascending input_order, last wins; departure-time
updates do not apply.

Expected checkpoint: first valid estimated departure; target is the request
effective at that checkpoint. A checkpoint before first input is not evaluable
and is excluded with its count reported. The main cohort remains frozen.

Static LP is explicitly an offline full-information oracle. Its replay is
initialized with the post-hoc target at arrival and ignores submission updates.
Online environments retain the original input-release policy. All use the
same event engine, aggregate capacity and ACN-Sim device executor.

Static LP uses ceil(real arrival) to floor(real departure), with complete slots.
Online execution can be interrupted by a real departure inside a slot.
The static planned optimum is therefore not automatically a lower bound on
online actual service shortfall. No such bound or formal savings claim is made.

Peak_kW is the maximum actual piecewise power. Grid-average peak is reported
separately. Zero-load time is retained. Fees exclude the optimization peak term.

RH trajectories are reused after source/input hash checks and independent
trajectory audits. Their original controller times are preserved.
Static offline solve time (budget 120 s) is separate from replay decision time.
Uncontrolled has no future plan; its plan-change diagnostic is NA.
Static's fixed future plan has zero revision. TOU and RH revisions compare
common future slots, with absent sparse entries interpreted as zero.
"""
(OUT / "integration_policy.md").write_text(policy, encoding="utf-8")
tracked = [
    Path(__file__), ROOT / "src/metrics.py",
    ROOT / "src/baselines.py", ROOT / "src/rolling_horizon.py",
    ROOT / "src/optimization_model.py", ROOT / "src/simulator.py",
    ROOT / "src/acnsim_adapter.py", ROOT / "src/tariff.py",
    ROOT / "configs/normalization.yaml", ROOT / "configs/station.yaml",
    ROOT / "configs/splits.yaml", ROOT / "configs/tariff.yaml",
    DATA / "sessions_with_split.csv", DATA / "environment_events.csv",
    DATA / "input_release_events.csv", DATA / "user_updates.csv",
    RH / "verification.json",
]
result = {
    "status": "passed", "check_count": len(passed), "checks": passed,
    "rh_reused": True, "rh_source": str(RH),
    "scope": "stage 3 development integration, not formal test results",
    "comparison": comparisons,
    "source_and_input_sha256": {
        str(p.relative_to(ROOT)): digest(p) for p in tracked},
    "output_artifact_sha256": artifact_hashes,
    "wall_seconds": perf_counter() - started,
}
(OUT / "verification.json").write_text(
    json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({
    "status": result["status"], "check_count": len(passed), "checks": passed,
    "rh_reused": True, "wall_seconds": result["wall_seconds"],
}, ensure_ascii=False, indent=2))
columns = [
    "cohort", "method", "cost_USD", "delivered_kwh", "unmet_kwh",
    "energy_satisfaction", "max_peak_kW", "uncompleted_sessions",
    "expected_checkpoint_evaluable_sessions", "expected_departure_uncompleted_rate",
    "fallback_count", "active_P95_seconds", "offline_solve_wall_seconds",
    "plan_change_mean_kW_sum",
]
print("\n=== Development integration comparison ===")
print(table[columns].to_string(index=False))
print("\nFull results:", OUT / "verification.json")
print("Policy:", OUT / "integration_policy.md")
'''

path = ROOT / "experiments/run_all.py"
text = source.lstrip()
compile(text, str(path), "exec")
if path.exists() and path.read_text(encoding="utf-8") != text:
    raise FileExistsError(f"保留已有不同文件，未覆盖：{path}")
path.write_text(text, encoding="utf-8")

print("开始统一集成，复用已有 RH-LP 轨迹。", flush=True)
output = LOG / "stdout_stderr.txt"
with output.open("w", encoding="utf-8") as stream:
    try:
        result = subprocess.run(
            [sys.executable, str(path), str(LOG)],
            cwd=ROOT, stdout=stream, stderr=subprocess.STDOUT,
            text=True, timeout=900,
        )
        code = result.returncode
    except subprocess.TimeoutExpired:
        code = "TIMEOUT"

print("解释器：", sys.executable)
print("退出码：", code)
print("证据目录：", LOG)
print(output.read_text(encoding="utf-8"))

开始统一集成，复用已有 RH-LP 轨迹。
解释器： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/.envs/ev-charging/bin/python
退出码： 1
证据目录： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/results/logs/stage3_integration_20261112T090634128692Z
/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/.envs/ev-charging/lib/python3.11/site-packages/acnportal/acnsim/base.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Integrating main_at_arrival/Uncontrolled
Integrating main_at_arrival/TOU_Greedy
Integrating main_at_arrival/Static_LP
Integrating main_at_arrival/RH_LP
Integrating supplement_after_submission/Uncontrolled
Integrating supplement_after_submission/TOU_Greedy
Integrating supplement_after_submission/Static_LP
Integrating supplement_after_submission/RH_LP
Traceb

In [18]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import shutil
import sys

ROOT = Path("/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
LOG = ROOT / "results/logs" / f"stage3_timestamp_fix_{stamp}"
LOG.mkdir(parents=True, exist_ok=False)

script = ROOT / "experiments/run_all.py"
original = script.read_text(encoding="utf-8")
old = 'pd.to_datetime(trace["start"], utc=True)'
new = 'pd.to_datetime(trace["start"], utc=True, format="ISO8601", errors="raise")'

if original.count(old) == 1:
    revised = original.replace(old, new)
elif new in original:
    revised = original
else:
    raise RuntimeError("源码与预期不同，未修改，请回传此输出。")

test_source = r'''
from pathlib import Path
from io import StringIO
import json
import sys
import numpy as np
import pandas as pd

passed = []
def check(name, condition):
    if not condition:
        raise AssertionError(name)
    passed.append(name)

def parse(values):
    return pd.to_datetime(
        pd.Series(values), utc=True, format="ISO8601", errors="raise")

def same_nanoseconds(raw, parsed):
    expected = np.array([pd.Timestamp(v).value for v in raw], dtype="int64")
    return np.array_equal(expected, parsed.array.asi8)

raw = [
    "2019-05-06 17:44:59+00:00",
    "2019-05-06 17:44:59.999999999+00:00",
    "2019-05-06T17:45:00.000001+00:00",
]
check("mixed precision preserves exact nanoseconds",
      same_nanoseconds(raw, parse(raw)))

repeated_hour = parse([
    "2019-11-03T01:30:00-07:00",
    "2019-11-03T01:30:00-08:00",
])
check("DST repeated hour remains two distinct instants",
      (repeated_hour.iloc[1] - repeated_hour.iloc[0]).value
      == 3_600_000_000_000)

frame = pd.read_csv(sys.argv[2], usecols=["start", "end"])
for column in ["start", "end"]:
    parsed = parse(frame[column])
    check(f"actual CSV {column} preserves nanoseconds",
          same_nanoseconds(frame[column], parsed))

parsed = parse(frame["start"])
buffer = StringIO()
pd.DataFrame({"stamp": parsed}).to_csv(buffer, index=False)
buffer.seek(0)
roundtrip = parse(pd.read_csv(buffer)["stamp"])
check("CSV roundtrip preserves nanoseconds",
      np.array_equal(parsed.array.asi8, roundtrip.array.asi8))

try:
    parse(["not-a-timestamp"])
except ValueError:
    check("invalid timestamp still raises", True)
else:
    raise AssertionError("Invalid timestamp silently accepted")

result = {"status": "passed", "check_count": len(passed), "checks": passed}
Path(sys.argv[1]).write_text(
    json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(result, ensure_ascii=False, indent=2))
'''

test_path = ROOT / "tests/test_trajectory_timestamps.py"
test_text = test_source.lstrip()
compile(revised, str(script), "exec")
compile(test_text, str(test_path), "exec")
if test_path.exists() and test_path.read_text(encoding="utf-8") != test_text:
    raise FileExistsError(f"保留已有不同文件，未覆盖：{test_path}")

artifact = ROOT / (
    "results/logs/stage3_integration_20261112T090634128692Z/"
    "supplement_after_submission/RH_LP/session_trajectory.csv"
)
assert artifact.is_file(), f"待检查的真实轨迹不存在：{artifact}"

shutil.copy2(script, LOG / "run_all_before.py")
script.write_text(revised, encoding="utf-8")
test_path.write_text(test_text, encoding="utf-8")

integration_dir = LOG / "integration"
integration_dir.mkdir()

jobs = [
    (
        "timestamp_regression",
        [sys.executable, str(test_path),
         str(LOG / "timestamp_verification.json"), str(artifact)],
        180,
    ),
    (
        "integration",
        [sys.executable, str(script), str(integration_dir)],
        900,
    ),
]

print("解释器：", sys.executable)
print("证据目录：", LOG)

for name, command, timeout in jobs:
    output = LOG / f"{name}_stdout_stderr.txt"
    with output.open("w", encoding="utf-8") as stream:
        try:
            result = subprocess.run(
                command, cwd=ROOT, stdout=stream,
                stderr=subprocess.STDOUT, text=True, timeout=timeout)
            code = result.returncode
        except subprocess.TimeoutExpired:
            code = "TIMEOUT"
    print(f"\n=== {name}：退出码 {code} ===")
    print(output.read_text(encoding="utf-8"))
    if code != 0:
        print("本轮停止，保留全部日志。")
        break

解释器： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/.envs/ev-charging/bin/python
证据目录： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/results/logs/stage3_timestamp_fix_20261112T091146144764Z

=== timestamp_regression：退出码 0 ===
{
  "status": "passed",
  "check_count": 6,
  "checks": [
    "mixed precision preserves exact nanoseconds",
    "DST repeated hour remains two distinct instants",
    "actual CSV start preserves nanoseconds",
    "actual CSV end preserves nanoseconds",
    "CSV roundtrip preserves nanoseconds",
    "invalid timestamp still raises"
  ]
}


=== integration：退出码 0 ===
/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/.envs/ev-charging/lib/python3.11/site-packages/acnportal/acnsim/base.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  im

In [19]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import csv
import shutil
import yaml

ROOT = Path("/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization")
FINAL = ROOT / (
    "results/logs/stage3_timestamp_fix_20261112T091146144764Z/integration"
)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUT = ROOT / "results/logs" / f"stage3_acceptance_{stamp}"
OUT.mkdir(parents=True, exist_ok=False)

def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

result = json.loads((FINAL / "verification.json").read_text(encoding="utf-8"))
assert result["status"] == "passed" and result["check_count"] == 64
assert len(result["comparison"]) == 8

index = []
for relative, expected in result["source_and_input_sha256"].items():
    path = ROOT / relative
    assert path.is_file() and sha(path) == expected, f"源文件变化：{relative}"
    index.append(["source_or_input", relative, expected])

for relative, expected in result["output_artifact_sha256"].items():
    path = FINAL / relative
    assert path.is_file() and sha(path) == expected, f"结果文件变化：{relative}"
    index.append(["integration_output", str(path.relative_to(ROOT)), expected])

evidence = [
    ("事件环境收尾", "stage3_environment_acceptance_20261112T074645020514Z/verification.json", 43),
    ("极小功率回归", "stage3_completion_fix_20261112T080048207614Z/completion_regression_verification.json", 7),
    ("事件回归", "stage3_completion_fix_20261112T080048207614Z/event_regression_verification.json", 22),
    ("执行回归", "stage3_completion_fix_20261112T080048207614Z/execution_regression_verification.json", 13),
    ("适配回归", "stage3_completion_fix_20261112T080048207614Z/adapter_regression_verification.json", 20),
    ("基线回归", "stage3_completion_fix_20261112T080048207614Z/baseline_regression_verification.json", 19),
    ("开发集基线回放", "stage3_completion_fix_20261112T080048207614Z/baseline_replay/verification.json", 37),
    ("RH合成检查", "stage3_rolling_synthetic_20261112T081433785555Z/verification.json", 17),
    ("RH动态回放", "stage3_rolling_replay_20261112T082426223430Z/verification.json", 22),
    ("指标手算检查", "stage3_metrics_20261112T085600815670Z/verification.json", 15),
    ("时间精度回归", "stage3_timestamp_fix_20261112T091146144764Z/timestamp_verification.json", 6),
    ("四方法集成", "stage3_timestamp_fix_20261112T091146144764Z/integration/verification.json", 64),
]

evidence_lines = []
for name, relative, count in evidence:
    path = ROOT / "results/logs" / relative
    record = json.loads(path.read_text(encoding="utf-8"))
    assert record["status"] == "passed" and record["check_count"] == count, name
    project_path = str(path.relative_to(ROOT))
    index.append(["acceptance_evidence", project_path, sha(path)])
    evidence_lines.append(f"| {name} | {count}/{count} | `{project_path}` |")

for row in result["comparison"]:
    assert row["fallback_count"] == 0
    assert row["max_power_violation_kW"] <= 1e-6
    assert row["max_energy_residual_kwh"] <= 1e-5
    assert row["connection_violation_segments"] == 0

norm = yaml.safe_load(
    (ROOT / "configs/normalization.yaml").read_text(encoding="utf-8"))
assert abs(norm["C_ref"] - 2.88) < 1e-12
assert abs(norm["P_ref"] - 6.656) < 1e-12

comparison_lines = []
for row in result["comparison"]:
    comparison_lines.append(
        f"| {row['cohort']} | {row['method']} | "
        f"{row['cost_USD']:.6f} | {row['delivered_kwh']:.6f} | "
        f"{row['unmet_kwh']:.6f} | {row['energy_satisfaction']:.4%} | "
        f"{row['max_peak_kW']:.6f} |"
    )

report = f"""# 阶段三：算法与仿真验收报告

验收日期：2026-11-12  
验收结论：**通过，范围为单站聚合连续功率实验模型。**

## 已完成内容

- 3.1：事件回放、输入按时释放、实际能量积分及 ACN-Sim 组件适配。
- 3.2：容量受限 Uncontrolled、TOU Greedy 及开发集基线验证。
- 3.3：双层 RH-LP、首步执行、实际日峰值反馈及 EDF 回退。
- 3.4：统一指标、四方法同会话与尾段评价、独立信息与物理审计。

主队列和补充队列分别评价。首周集成样本为 5 条与 198 条。
RH 动态回放共 680 次 LP 决策、零回退，活跃决策 P95 约 2.24/2.30 秒。
正式归一化为 C_ref={norm['C_ref']:.12g} USD、P_ref={norm['P_ref']:.12g} kW，
来自开发集严格主队列容量受限 Uncontrolled 的全部 49 个自然日，
含 15 个零值日，均采用全日中位数。

## 验收证据

下列检查包含回归与前序证据核对，不将其简单相加称为独立测试总数。

| 验证项 | 通过数 | 证据 |
|---|---:|---|
{chr(10).join(evidence_lines)}

## 开发集首周集成结果

| 队列 | 方法 | 费用 USD | 实际交付 kWh | 缺口 kWh | 能量满足率 | 实际峰值 kW |
|---|---|---:|---:|---:|---:|---:|
{chr(10).join(comparison_lines)}

## 评价口径

主队列以到达时冻结请求评价；补充队列以真实离站前最后有效请求评价。
预计离站检查点取首次有效预计离站时刻，目标取该检查点已生效请求；
早于首次有效输入的检查点单列为不可评价，不入其分母。
同刻离站更新无效。逐车截断交付后计算满足率，超额不能抵消其他车辆缺口。

实际分段功率峰值与 15 分钟平均功率峰值分别输出。
费用只计实际电能费用，不含优化峰值代理项。

Static LP 为完整信息离线参考，使用事后目标及真实连接完整时段；
在线方法可能在时段内遭遇真实离站。因此不自动将静态计划缺口
解释为在线实际缺口的理论下界。离线求解耗时与在线决策耗时分开报告。

RH 轨迹经代码、输入哈希核验及独立轨迹审计后复用，
原始决策耗时保留。聚合网络采用实验容量，未验证原 Caltech 支路相位网络。

## 结果解释及后续工作

补充队列 RH-LP 的交付量低于规则基线，较低费用不能直接称为节费优势。
本表是开发集集成验证，不是正式测试结果，不据此预设算法性能结论。

已修复充满时长溢出和 CSV 混合时间精度解析问题，失败日志保留。
下一阶段为 4.1：验证集参数选择、正式配置冻结及独立测试实验。
全流程干净环境复现留待阶段五。

详细策略：`{str((FINAL / 'integration_policy.md').relative_to(ROOT))}`  
完整结果：`{str((FINAL / 'verification.json').relative_to(ROOT))}`
"""

report_dir = ROOT / "report"
report_dir.mkdir(exist_ok=True)
report_path = report_dir / "stage3_acceptance.md"
index_path = report_dir / "stage3_artifact_index.csv"

for path in [report_path, index_path]:
    if path.exists():
        shutil.copy2(path, OUT / f"before_{path.name}")

report_path.write_text(report, encoding="utf-8")
index.append(["acceptance_report", str(report_path.relative_to(ROOT)), sha(report_path)])
with index_path.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream)
    writer.writerow(["role", "project_relative_path", "sha256"])
    writer.writerows(index)

acceptance = {
    "status": "passed",
    "stage": 3,
    "tasks": ["3.1", "3.2", "3.3", "3.4"],
    "evidence_records_verified": len(evidence),
    "indexed_artifacts": len(index),
    "report": str(report_path),
    "report_sha256": sha(report_path),
    "artifact_index": str(index_path),
    "artifact_index_sha256": sha(index_path),
    "integration_evidence": str(FINAL),
    "C_ref": norm["C_ref"],
    "P_ref": norm["P_ref"],
    "formal_test_experiments_started": False,
}
(OUT / "verification.json").write_text(
    json.dumps(acceptance, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(acceptance, ensure_ascii=False, indent=2))
print("归档证据目录：", OUT)

{
  "status": "passed",
  "stage": 3,
  "tasks": [
    "3.1",
    "3.2",
    "3.3",
    "3.4"
  ],
  "evidence_records_verified": 12,
  "indexed_artifacts": 70,
  "report": "/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/report/stage3_acceptance.md",
  "report_sha256": "9a15bcd90ce19a4732e99c69453aa5fc83ba013cfe6acec093c8b0c0f501e980",
  "artifact_index": "/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/report/stage3_artifact_index.csv",
  "artifact_index_sha256": "ffc0a0d13db243c2d302111b6ab4005a450a8b4f238e892c890423419113aa85",
  "integration_evidence": "/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/results/logs/stage3_timestamp_fix_20261112T091146144764Z/integration",
  "C_ref": 2.8800000000000003,
  "P_ref": 6.656,
  "formal_test_experiments_started": false
}
归档证据目录： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/results/logs/stage3_acceptance_20261112T091641423634Z
